# libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import os
from scipy.stats import pearsonr
import math

#load plot and tree data

In [ ]:
from pathlib import Path

# Change only this root directory when adapting the workflow.
PROJECT_ROOT = Path("/content/drive/MyDrive/AlphaEarth_MBI_Remote_Forests")

DATA_DIR = PROJECT_ROOT / "data"
FIELD_DIR = DATA_DIR / "field"
SPATIAL_DIR = DATA_DIR / "spatial"
INTERMEDIATE_DIR = PROJECT_ROOT / "intermediate"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Restricted/user-supplied inputs
TREE_DATA_PATH = FIELD_DIR / "tree_measurements.xlsx"
FIELD_PLOT_PATH = FIELD_DIR / "field_plot_attributes.csv"
STUDY_PLOTS_PATH = SPATIAL_DIR / "study_area_plots.gpkg"
MANAGEMENT_REGIONS_PATH = SPATIAL_DIR / "management_regions.shp"

# Intermediate/public workflow products
EMBEDDING_BATCH_DIR = INTERMEDIATE_DIR / "embedding_batches"
MODEL_INPUT_PATH = INTERMEDIATE_DIR / "model_input_aef_dem_bioclim.csv"
PREDICTION_OUTPUT_PATH = OUTPUT_DIR / "forest_structure_predictions.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)


### Expected user-supplied data

This public notebook does not distribute the restricted NFI inputs. The workflow expects (1) tree-level measurements with a plot identifier, (2) plot-level field attributes and corrected coordinates, (3) the complete study-area plot layer, and (4) a management-region layer identifying Gilan, Nowshahr, Sari, and Golestan for spatial block cross-validation. Adapt column names in the loading/preparation section if your authorized data use a different schema.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load restricted field inputs from the public configuration paths.
tree_data = pd.read_excel(TREE_DATA_PATH)

plot_data = pd.read_csv(FIELD_PLOT_PATH)


In [ ]:
plot_data = plot_data.rename(columns={'PLOTID_org': 'PLOTID'})

tree_data = tree_data[tree_data['HFI'].astype(str).str.strip().eq('HFI6')].copy()

In [ ]:
n_trees_calculated = tree_data.groupby('PLOTID').size().reset_index(name='n_trees')
n_trees_calculated['n_trees'] = n_trees_calculated['n_trees'] * 10

plot_data = plot_data.merge(n_trees_calculated, on='PLOTID', how='left')
plot_data

In [ ]:
from scipy import stats
import numpy as np

# Define the columns of interest
columns_of_interest = ['max_h', 'n_trees', 'Volume']

results = {}

for col in columns_of_interest:
    if col in plot_data.columns:
        # Drop NA values for the calculation to avoid errors
        data = plot_data[col].dropna()

        mean_val = data.mean()
        std_val = data.std()
        count_val = len(data)

        # Calculate 95% Confidence Interval
        # For a 95% CI, alpha = 0.05, so alpha/2 = 0.025 (or 1 - 0.025 = 0.975 for the upper tail)
        # Degrees of freedom (df) = count - 1
        if count_val > 1:
            sem = std_val / np.sqrt(count_val)  # Standard Error of the Mean
            confidence_interval = stats.t.interval(0.95, df=count_val-1, loc=mean_val, scale=sem)
        else:
            confidence_interval = (np.nan, np.nan) # Cannot calculate CI with less than 2 data points

        results[col] = {
            'mean': mean_val,
            'std': std_val,
            'count': count_val,
            '95% CI': confidence_interval
        }
    else:
        print(f"Column '{col}' not found in plot_data.")

# Print the results
for col, metrics in results.items():
    print(f"\n--- {col} ---")
    print(f"Mean: {metrics['mean']:.2f}")
    print(f"Standard Deviation: {metrics['std']:.2f}")
    print(f"Count: {metrics['count']}")
    print(f"95% Confidence Interval: ({metrics['95% CI'][0]:.2f}, {metrics['95% CI'][1]:.2f})")

In [ ]:
import geopandas as gpd

# Construct the full path to the GeoPackage file on Google Drive
gpkg_path = STUDY_PLOTS_PATH

# Load the GeoPackage file into a GeoDataFrame
study_plots = gpd.read_file(gpkg_path)

# Display the head of the GeoDataFrame
print(study_plots.head())

In [ ]:
import pandas as pd
import geopandas as gpd


# ============================================================
# 3. Prepare ID fields
# ============================================================

study_plots["code"] = study_plots["code"].astype(str)
plot_data["PLOTID"] = plot_data["PLOTID"].astype(str)

# Remove accidental spaces
study_plots["code"] = study_plots["code"].str.strip()
plot_data["PLOTID"] = plot_data["PLOTID"].str.strip()

# ============================================================
# 4. Merge while keeping ALL records from study_plots
# ============================================================

merged_plots = study_plots.merge(
    plot_data,
    left_on="PlotID",
    right_on="PLOTID",
    how="left"
)

# ============================================================
# 5. Add measured / unmeasured status
# ============================================================

merged_plots["plot_status"] = merged_plots["PLOTID"].notna().map({
    True: "measured",
    False: "unmeasured"
})

# ============================================================
# 6. Check results
# ============================================================

print("Number of records in study_plots:", len(study_plots))
print("Number of records after merge:", len(merged_plots))

print("\nPlot status:")
print(merged_plots["plot_status"].value_counts())

print("\nNumber of measured plots:", (merged_plots["plot_status"] == "measured").sum())
print("Number of unmeasured plots:", (merged_plots["plot_status"] == "unmeasured").sum())

# Check duplicated IDs after merge
print("\nDuplicated code values after merge:")
print(merged_plots["code"].duplicated().sum())



In [ ]:
merged_plots_final_latlon = merged_plots.copy()


# Rename 'xlat' to 'Longitude' and 'y.lat' to 'Latitude'
# These columns appear to contain the desired geographic coordinates from the source data
merged_plots_final_latlon = merged_plots_final_latlon.rename(columns={'x_coord': 'Longitude', 'y_coord': 'Latitude'})

print("Shape of the new merged_plots_final_latlon DataFrame:", merged_plots_final_latlon.shape)
display(merged_plots_final_latlon.head())

In [ ]:
# Number of valid (non-null) values per column
valid_values = merged_plots_final_latlon.notnull().sum()

# Sort descending
valid_values = valid_values.sort_values(ascending=False)

print("Valid values per column:")
display(valid_values)

In [ ]:

columns_to_drop = [
    'rdf', 'x', 'y', 'zone', 'xlat', 'y.lat',
    'geometry_x', 'geometry_y'
]

# Drop the specified columns, ignoring errors if a column doesn't exist
# Longitude and Latitude are NOT dropped here, as they are needed for GEE processing
# and will be handled for duplicates in the next cell.
merged_plots_final_latlon = merged_plots_final_latlon.drop(
    columns=columns_to_drop,
    errors='ignore'
)

# --- FIX: Handle duplicate 'code' values in merged_plots_final_latlon ---
# The 'code' column in merged_plots_final_latlon can have duplicates (18 identified earlier).
# When this DataFrame is later merged with 'merged_embeddings_df' (which has unique 'code's from GEE),
# these duplicates cause an expansion in rows (e.g., 4086 -> 4120).
# To ensure a 1:1 merge for 'code', we remove duplicates, keeping the first occurrence.
initial_rows = merged_plots_final_latlon.shape[0]
merged_plots_final_latlon.drop_duplicates(subset=['code'], keep='first', inplace=True)
dropped_duplicates_count = initial_rows - merged_plots_final_latlon.shape[0]

print(f"Removed {dropped_duplicates_count} duplicate 'code' entries from merged_plots_final_latlon.")
print("Columns dropped from merged_plots_final_latlon.")
print("New shape of merged_plots_final_latlon:", merged_plots_final_latlon.shape)
display(merged_plots_final_latlon.head())

# GEE config

In [ ]:
from google.colab import auth
auth.authenticate_user(project_id='YOUR_GCP_PROJECT_ID')

In [ ]:

import ee
ee.Authenticate()  # Run this to authenticate your account
ee.Initialize()

# Geometry for field data

In [ ]:

import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping
import geemap
import numpy as np

# --- Points -> EE FeatureCollection ---
df = merged_plots_final_latlon.copy()

# Resolve duplicate 'Longitude' and 'Latitude' columns if they exist.
# This happens because 'merged_plots' has 'x_coord', 'y_coord' (from study_plots)
# and 'Longitude', 'Latitude' (from plot_data).
# In cell 2f2b6ef1, 'x_coord' and 'y_coord' are renamed to 'Longitude' and 'Latitude',
# leading to duplicate column names if the 'Longitude' and 'Latitude' from 'plot_data'
# were not explicitly dropped earlier.
# The 'Longitude'/'Latitude' with more non-null values are preferred (from 'x_coord'/'y_coord').

# Handle duplicate 'Longitude' columns
if 'Longitude' in df.columns[df.columns.duplicated(keep=False)].unique():
    print("Detected duplicate 'Longitude' columns. Resolving by keeping the one with most non-nulls.")
    lon_cols_for_selection = df.loc[:, 'Longitude']
    non_null_counts = [lon_cols_for_selection.iloc[:, i].notna().sum() for i in range(lon_cols_for_selection.shape[1])]
    best_lon_col_idx_in_dups_df = np.argmax(non_null_counts)
    df['Longitude_resolved'] = lon_cols_for_selection.iloc[:, best_lon_col_idx_in_dups_df]
    df = df.drop(columns=[col for col in df.columns if col == 'Longitude'], errors='ignore')
    df = df.rename(columns={'Longitude_resolved': 'Longitude'})

# Handle duplicate 'Latitude' columns
if 'Latitude' in df.columns[df.columns.duplicated(keep=False)].unique():
    print("Detected duplicate 'Latitude' columns. Resolving by keeping the one with most non-nulls.")
    lat_cols_for_selection = df.loc[:, 'Latitude']
    non_null_counts = [lat_cols_for_selection.iloc[:, i].notna().sum() for i in range(lat_cols_for_selection.shape[1])]
    best_lat_col_idx_in_dups_df = np.argmax(non_null_counts)
    df['Latitude_resolved'] = lat_cols_for_selection.iloc[:, best_lat_col_idx_in_dups_df]
    df = df.drop(columns=[col for col in df.columns if col == 'Latitude'], errors='ignore')
    df = df.rename(columns={'Latitude_resolved': 'Latitude'})

# If you want to preserve the existing geometry column from merged_plots_final:
# This block is likely not needed if starting from merged_plots_final_latlon
# as 'geometry' should have been dropped.
if "geometry" in df.columns:
    df = df.rename(columns={"geometry": "geometry_orig"})

# Rename 'y.lat' column to a valid Python identifier for compatibility with _asdict()
# This block is likely not needed if starting from merged_plots_final_latlon
# as 'y.lat' should have been renamed to 'Latitude'.
if 'y.lat' in df.columns:
    df = df.rename(columns={'y.lat': 'y_lat_prop'})

df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
df["Latitude"]  = pd.to_numeric(df["Latitude"],  errors="coerce")
df = df.dropna(subset=["Longitude", "Latitude"]).reset_index(drop=True)

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326"
)

print(f"Number of points in local GeoDataFrame (gdf): {len(gdf)}")

def _to_py_scalar(v):
    if pd.isna(v):
        return None
    return v.item() if hasattr(v, "item") else v

# Only include 'code' as a property for the Earth Engine FeatureCollection
# to avoid duplicating columns already present in merged_embeddings_df.
prop_cols = ['code']

features = [
    ee.Feature(
        ee.Geometry.Point([float(r.Longitude), float(r.Latitude)]),
        {k: _to_py_scalar(r._asdict()[k]) for k in prop_cols}
    )
    for r in gdf.itertuples(index=False)
]
table_field_data = ee.FeatureCollection(features)

print(f"Number of features in Earth Engine FeatureCollection (table_field_data): {table_field_data.size().getInfo()}")
print(table_field_data.first().getInfo())

# --- Polygon shapefile -> EE FeatureCollection ---
shapefile_path = MANAGEMENT_REGIONS_PATH
study_area = gpd.read_file(shapefile_path)

if study_area.crs is None:
    study_area = study_area.set_crs("EPSG:4326")
elif study_area.crs.to_epsg() != 4326:
    study_area = study_area.to_crs(epsg=4326)

# attempt to fix invalid geoms rather than dropping
study_area["geometry"] = study_area["geometry"].buffer(0)
study_area = study_area[study_area.is_valid & ~study_area.geometry.is_empty]

def gdf_to_ee(gdf):
    feats = []
    for r in gdf.itertuples(index=False):
        geom = ee.Geometry(mapping(r.geometry))
        props = pd.Series(r._asdict()).drop("geometry").to_dict()
        feats.append(ee.Feature(geom, props))
    return ee.FeatureCollection(feats)

study_area_fc = gdf_to_ee(study_area)
aoi_geom = study_area_fc.geometry()

# --- Map ---
m = geemap.Map()
m.add_basemap("SATELLITE")

m.addLayer(study_area_fc, {"color": "red", "fillColor": "yellow", "fillOpacity": 0.3}, "Study Area Polygon")
m.addLayer(table_field_data, {"color": "cyan"}, "Field plots")

m.centerObject(study_area_fc, 10)
m

# Google embeddings

In [ ]:
dataset = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

aoi_geom = study_area_fc.geometry()

# Pick the annual image for 2023 and mosaic all images that intersect the AOI
embedding2023 = (dataset
  .filterDate('2023-01-01', '2024-01-01')
  .filterBounds(aoi_geom)
  .mosaic() # Changed from .first() to .mosaic() to ensure full coverage
  .clip(aoi_geom)
  .toFloat()
)

# To avoid "User memory limit exceeded" when calling getInfo() on a potentially large
# clipped image, get the band names from a less computationally intensive object.
# The band names are consistent across images in this collection for a given year,
# and are not affected by clipping or conversion to float.
temp_image_for_band_names = dataset.filterDate('2023-01-01', '2024-01-01').first()

band_names = []
if temp_image_for_band_names:
    band_names = temp_image_for_band_names.bandNames().getInfo()
else:
    print("WARNING: No embedding image found for 2023 to retrieve band names.")

print(len(band_names), band_names[:10], "...", band_names[-3:])

In [ ]:
# To avoid memory issues with large exports, we'll split the points into batches and export them incrementally.
# Adjust `batch_size` based on how many points Earth Engine can handle per task without OOM errors.
batch_size = 500  # You can adjust this value if errors persist or if you want larger batches

points_list = table_field_data.toList(table_field_data.size()) # Changed from points_fc to table_field_data
num_points = table_field_data.size().getInfo() # Changed from points_fc to table_field_data

for i in range(0, num_points, batch_size):
    end_index = min(i + batch_size, num_points)
    batch_points = ee.FeatureCollection(points_list.slice(i, end_index))

    # Sample embeddings for the current batch of points
    sampled_emb_batch = embedding2023.sampleRegions(
        collection=batch_points,
        scale=10,
        geometries=False,
        tileScale=4
    )

    # Define a unique description and filename prefix for each batch
    description = f'Embeddings2023_Batch_{i}_to_{end_index-1}'
    file_name_prefix = f'Embeddings2023_Batch_{i}_to_{end_index-1}'

    task = ee.batch.Export.table.toDrive(
        collection=sampled_emb_batch,
        description=description,
        folder='AlphaEarth_MBI_exports',
        fileNamePrefix=file_name_prefix,
        fileFormat='CSV'
    )
    task.start()
    print(f'🚀 Export started for {description}')

print('All embedding export tasks initiated.')

In [ ]:
print(f"Number of points in table_field_data after filtering by embedding2023 geometry: {table_field_data.filterBounds(embedding2023.geometry()).size().getInfo()}")

In [ ]:
import geemap

m_map = geemap.Map()
m_map.add_basemap("SATELLITE")

m_map.addLayer(study_area_fc, {"color": "red", "fillColor": "yellow", "fillOpacity": 0.3}, "Study Area Polygon")
m_map.addLayer(table_field_data, {"color": "cyan"}, "All Field Plots (table_field_data)")

# Filter table_field_data by embedding2023 geometry
intersecting_table_field_data = table_field_data.filterBounds(embedding2023.geometry())
m_map.addLayer(intersecting_table_field_data, {"color": "lime"}, "Intersecting Field Plots")

# Add the clipped embedding2023 image directly to the map
m_map.addLayer(embedding2023, {'bands': ['A01', 'A16', 'A09'], 'min': -0.3, 'max': 0.3}, "Embedding2023 Clipped (A01,A16,A09)")

m_map.centerObject(study_area_fc, 10)
m_map

# Load RS and Field data

In [ ]:
import pandas as pd
import os

# Define the folder in Google Drive where the batches are saved
drive_folder = str(EMBEDDING_BATCH_DIR)

# Define parameters used in batch export
batch_size = 500
num_points = 4140 # Total number of points (from previous context)

all_batches_df = []

print("Reading and merging embedding batches...")

for i in range(0, num_points, batch_size):
    end_index = min(i + batch_size, num_points)
    file_name_prefix = f'Embeddings2023_Batch_{i}_to_{end_index-1}'
    csv_file_path = os.path.join(drive_folder, file_name_prefix + ".csv")

    if os.path.exists(csv_file_path):
        print(f"  - Found and reading: {file_name_prefix}.csv")
        try:
            batch_df = pd.read_csv(csv_file_path)
            all_batches_df.append(batch_df)
        except Exception as e:
            print(f"    Error reading {file_name_prefix}.csv: {e}")
    else:
        print(f"  - Warning: {file_name_prefix}.csv not found. Skipping.")

if all_batches_df:
    merged_embeddings_df = pd.concat(all_batches_df, ignore_index=True)
    print("\nSuccessfully merged all available batches.")
    print("Shape of the merged embeddings DataFrame:", merged_embeddings_df.shape)
    display(merged_embeddings_df.head())
else:
    print("No embedding batch files were found or successfully read.")
merged_embeddings_df.shape

In [ ]:
import pandas as pd

# Ensure 'code' columns are of string type for consistent merging
merged_embeddings_df['code'] = merged_embeddings_df['code'].astype(str)
merged_plots_final_latlon['code'] = merged_plots_final_latlon['code'].astype(str)

# Perform a left merge. This will add suffixes for duplicate columns (other than 'code')
# keeping all rows from merged_embeddings_df
field_embd = pd.merge(
    merged_embeddings_df,
    merged_plots_final_latlon,
    on='code',
    how='left',
    suffixes=('', '_drop') # Add _drop suffix to columns from the right DataFrame
)

# Identify and drop columns ending with '_drop' (duplicates from the right DataFrame)
drop_cols = [col for col in field_embd.columns if col.endswith('_drop')]
field_embd = field_embd.drop(columns=drop_cols)

print(f"Shape of field_embd after merging and dropping duplicates: {field_embd.shape}")
display(field_embd.head())
print(f"Shape of merged_embeddings_df before dropping duplicates: {field_embd.shape}")


In [ ]:
import ee
import pandas as pd # Ensure pandas is imported if not already in global scope
import math # Ensure math is imported if not already in global scope
import geemap # Ensure geemap is imported if not already in global scope

ee.Initialize()

# ============================================================
# LOAD SRTM DEM 30 m
# ============================================================

dem = (
    ee.Image('USGS/SRTMGL1_003')
    .select('elevation')
    .clip(aoi_geom)
)

# ============================================================
# TERRAIN VARIABLES
# ============================================================

elevation = dem.rename('elevation')

slope = ee.Terrain.slope(dem).rename('slope')
aspect = ee.Terrain.aspect(dem).rename('aspect')

northness = (
    aspect.multiply(math.pi / 180)
    .cos()
    .rename('northness')
)

eastness = (
    aspect.multiply(math.pi / 180)
    .sin()
    .rename('eastness')
)

tpi = (
    dem.subtract(
        dem.focal_mean(
            radius=30,
            kernelType='circle',
            units='meters'
        )
    )
    .rename('tpi')
)

# ============================================================
# CURVATURE
# ============================================================

laplacian_kernel = ee.Kernel.fixed(
    width=3,
    height=3,
    weights=[
        [0,  1, 0],
        [1, -4, 1],
        [0,  1, 0]
    ],
    x=1,
    y=1,
    normalize=False
)

curvature = dem.convolve(laplacian_kernel).rename('curvature')

# ============================================================
# COMBINE TERRAIN VARIABLES
# ============================================================

terrain_features = ee.Image.cat([
    elevation,
    slope,
    aspect,
    northness,
    eastness,
    tpi,
    curvature
])

# ============================================================
# SAMPLE TERRAIN VARIABLES AT PLOTS
# ============================================================

sampled_dem_features = terrain_features.sampleRegions(
    collection=table_field_data,
    scale=30,
    geometries=False,
    tileScale=4
)

dem_features_df = geemap.ee_to_df(sampled_dem_features)

print("DEM extraction finished.")
print(dem_features_df.shape)
display(dem_features_df.head())

# ============================================================
# MERGE WITH EMBEDDINGS
# ============================================================

dem_features_df['code'] = dem_features_df['code'].astype(str)
field_embd['code'] = field_embd['code'].astype(str)

# --- FIX: Handle duplicate 'code' values in dem_features_df ---
# The dem_features_df might contain duplicate 'code' entries if some points
# in table_field_data had issues during sampling or if the GEE sampling
# process introduced duplicates for a given code. A left merge would
# expand the rows. We ensure uniqueness here before merging.
initial_dem_rows = dem_features_df.shape[0]
dem_features_df.drop_duplicates(subset=['code'], keep='first', inplace=True)
dropped_dem_duplicates_count = initial_dem_rows - dem_features_df.shape[0]

if dropped_dem_duplicates_count > 0:
    print(f"Removed {dropped_dem_duplicates_count} duplicate 'code' entries from dem_features_df before merging.")

merged_embeddings_field_dem = pd.merge(
    field_embd,
    dem_features_df,
    on='code',
    how='left'
)

print("Merged dataframe shape:")
print(merged_embeddings_field_dem.shape)

# ============================================================
# SAVE OUTPUT
# ============================================================

output_csv_path = str(INTERMEDIATE_DIR / "aef_field_dem.csv")

merged_embeddings_field_dem.to_csv(output_csv_path, index=False)

print(f"Saved: {output_csv_path}")

# ============================================================
# VERIFY
# ============================================================

confirm_df = pd.read_csv(output_csv_path)

print("Verification shape:")
print(confirm_df.shape)
display(confirm_df.head())

In [ ]:
import ee
import geemap
import pandas as pd

ee.Initialize()

print("Starting BIOCLIM extraction...")

bioclim = ee.Image("WORLDCLIM/V1/BIO").clip(aoi_geom)

bio_band_names = [
    "bio01", "bio02", "bio03", "bio04", "bio05",
    "bio06", "bio07", "bio08", "bio09", "bio10",
    "bio11", "bio12", "bio13", "bio14", "bio15",
    "bio16", "bio17", "bio18", "bio19"
]

bioclim = bioclim.rename(bio_band_names)

sampled_bioclim = bioclim.sampleRegions(
    collection=table_field_data,
    scale=1000,
    geometries=False,
    tileScale=4
)

bioclim_df = geemap.ee_to_df(sampled_bioclim)

print("BIOCLIM dataframe shape:")
print(bioclim_df.shape)
display(bioclim_df.head())

# Merge BIOCLIM with DEM + embeddings dataframe
bioclim_df["code"] = bioclim_df["code"].astype(str)
confirm_df["code"] = confirm_df["code"].astype(str)

merged_embeddings_field_dem_bioclim = pd.merge(
    confirm_df,
    bioclim_df,
    on="code",
    how="left"
)

print("Final merged dataframe shape:")
print(merged_embeddings_field_dem_bioclim.shape)

output_csv_path = str(MODEL_INPUT_PATH)

merged_embeddings_field_dem_bioclim.to_csv(
    output_csv_path,
    index=False
)

print(f"Saved: {output_csv_path}")

check_df = pd.read_csv(output_csv_path)

print("Verification shape:")
print(check_df.shape)
display(check_df.head())

# Full dataset for modelimg

In [ ]:

merged_embeddings_field_dem_bioclim = pd.read_csv (str(MODEL_INPUT_PATH))
print("Columns in merged_embeddings_field_dem_bioclim:", merged_embeddings_field_dem_bioclim.columns.tolist())
merged_embeddings_field_dem_bioclim.shape

In [ ]:
missing_values_count = merged_embeddings_field_dem_bioclim.isnull().sum()

# Filter to show only columns with missing values
missing_values_count = missing_values_count[missing_values_count > 0]

print("Missing values per column in merged_embeddings_field_dem_bioclim:")
if missing_values_count.empty:
    print("No missing values found.")
else:
    display(missing_values_count.sort_values(ascending=False))

# Calculate and display valid values count
valid_values_count = merged_embeddings_field_dem_bioclim.notnull().sum()

print("\nValid values per column in merged_embeddings_field_dem_bioclim:")
if valid_values_count.empty:
    print("No columns found.")
else:
    display(valid_values_count.sort_values(ascending=False))

# AOA

In [ ]:
# ============================================================
# Compact table for manuscript by restriction class
# ============================================================

import pandas as pd
import numpy as np

df = merged_embeddings_field_dem_bioclim.copy()

def summarize_variable(df_group, variable):
    x = df_group[variable].dropna()

    if len(x) == 0:
        return "NA"

    return (
        f"{x.mean():.1f} ± {x.std():.1f} "
        f"({x.min():.1f}–{x.max():.1f})"
    )

rows = []

# ------------------------------------------------------------
# 1) Measured plots
# ------------------------------------------------------------

measured_df = df[df["plot_status"] == "measured"]

rows.append({
    "Group": "Measured",
    "N": len(measured_df),
    "Elevation (m)": summarize_variable(measured_df, "elevation"),
    "Slope (°)": summarize_variable(measured_df, "slope")
})

# ------------------------------------------------------------
# 2) Unmeasured plots by restriction class
# ------------------------------------------------------------

unmeasured_df = df[df["plot_status"] == "unmeasured"]

for restriction in sorted(unmeasured_df["Restrictions"].dropna().unique()):

    restriction_df = unmeasured_df[
        unmeasured_df["Restrictions"] == restriction
    ]

    rows.append({
        "Group": f"Unmeasured - {restriction}",
        "N": len(restriction_df),
        "Elevation (m)": summarize_variable(restriction_df, "elevation"),
        "Slope (°)": summarize_variable(restriction_df, "slope")
    })

# ------------------------------------------------------------
# 3) All unmeasured plots
# ------------------------------------------------------------

rows.append({
    "Group": "Unmeasured - Total",
    "N": len(unmeasured_df),
    "Elevation (m)": summarize_variable(unmeasured_df, "elevation"),
    "Slope (°)": summarize_variable(unmeasured_df, "slope")
})

# ------------------------------------------------------------
# 4) Total dataset
# ------------------------------------------------------------

rows.append({
    "Group": "Total",
    "N": len(df),
    "Elevation (m)": summarize_variable(df, "elevation"),
    "Slope (°)": summarize_variable(df, "slope")
})

paper_table = pd.DataFrame(rows)

display(paper_table)

In [ ]:
# ============================================================
# AOA comparison figure by Restrictions
# Rows = Restrictions classes
# Columns = Topographic, Bioclimatic, Environmental variables
# ============================================================

import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ------------------------------------------------------------
# Adjustable font settings
# ------------------------------------------------------------

BASE_FONT_SIZE = 34
TITLE_FONT_SIZE = 34
AXIS_LABEL_SIZE = 36
TICK_LABEL_SIZE = 32
ROW_LABEL_SIZE = 38

LEGEND_FONT_SIZE = 44
LEGEND_MARKER_SIZE = 36   # increase if markers still look small

plt.rcParams.update({
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": TITLE_FONT_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "xtick.labelsize": TICK_LABEL_SIZE,
    "ytick.labelsize": TICK_LABEL_SIZE,
    "legend.fontsize": LEGEND_FONT_SIZE
})
# ------------------------------------------------------------
# PCA helper
# ------------------------------------------------------------

def get_pca_scores_by_restriction(data, features):

    sub = data[["plot_status", "Restrictions"] + features].copy()
    sub = sub.dropna(subset=features)

    measured = sub[sub["plot_status"] == "measured"].copy()
    unmeasured = sub[sub["plot_status"] == "unmeasured"].copy()

    scaler = StandardScaler()

    Xm = scaler.fit_transform(measured[features])
    Xu = scaler.transform(unmeasured[features])

    pca = PCA(n_components=2)

    Xm_pca = pca.fit_transform(Xm)
    Xu_pca = pca.transform(Xu)

    measured_scores = measured[["plot_status", "Restrictions"]].copy()
    measured_scores["PC1"] = Xm_pca[:, 0]
    measured_scores["PC2"] = Xm_pca[:, 1]

    unmeasured_scores = unmeasured[["plot_status", "Restrictions"]].copy()
    unmeasured_scores["PC1"] = Xu_pca[:, 0]
    unmeasured_scores["PC2"] = Xu_pca[:, 1]

    scores = pd.concat(
        [measured_scores, unmeasured_scores],
        axis=0,
        ignore_index=True
    )

    return scores, pca

# ------------------------------------------------------------
# Define feature lists
# ------------------------------------------------------------

topo_features = [
    'elevation',
    'slope',
    'aspect',
    'northness',
    'eastness',
    'tpi',
    'curvature'
]

bio_features = [
    "bio01", "bio02", "bio03", "bio04", "bio05",
    "bio06", "bio07", "bio08", "bio09", "bio10",
    "bio11", "bio12", "bio13", "bio14", "bio15",
    "bio16", "bio17", "bio18", "bio19"
]

# Extract AEF columns dynamically from df
AEF_COLS = [
    c for c in df.columns
    if len(c) == 3 and c[0] == "A" and c[1:].isdigit()
]
AEF_COLS = sorted(
    AEF_COLS,
    key=lambda x: int(x[1:]) if isinstance(x, str) and x[1:].isdigit() else 10**9
)

# ------------------------------------------------------------
# Calculate PCA spaces
# ------------------------------------------------------------

topo_scores, topo_pca = get_pca_scores_by_restriction(df, topo_features)
bio_scores, bio_pca = get_pca_scores_by_restriction(df, bio_features)
aef_scores, aef_pca = get_pca_scores_by_restriction(df, AEF_COLS) # New: PCA for AEF embeddings

datasets = [
    (topo_scores, topo_pca, "Topographic variables"),
    (bio_scores, bio_pca, "Bioclimatic variables"),
    (aef_scores, aef_pca, "AEF Embeddings") # New: AEF embeddings dataset
]

# ------------------------------------------------------------
# Restriction classes
# ------------------------------------------------------------

restriction_classes = sorted(
    df.loc[df["plot_status"] == "unmeasured", "Restrictions"]
    .dropna()
    .unique()
)

n_rows = len(restriction_classes)
n_cols = len(datasets) # Dynamically set n_cols based on the number of datasets

# ------------------------------------------------------------
# Create figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(12 * n_cols, 16 * n_rows), # Adjusted figsize for 3 columns
    squeeze=False
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

for row_idx, restriction in enumerate(restriction_classes):

    for col_idx, (scores, pca, title) in enumerate(datasets):

        ax = axes[row_idx, col_idx]

        measured = scores[scores["plot_status"] == "measured"]

        unmeasured_restriction = scores[
            (scores["plot_status"] == "unmeasured") &
            (scores["Restrictions"] == restriction)
        ]

        pc1_var = pca.explained_variance_ratio_[0] * 100
        pc2_var = pca.explained_variance_ratio_[1] * 100

        ax.scatter(
            measured["PC1"],
            measured["PC2"],
            s=45,
            alpha=0.55,
            label="Measured"
        )

        # Modified: Change label for unmeasured plots to just "Unmeasured"
        ax.scatter(
            unmeasured_restriction["PC1"],
            unmeasured_restriction["PC2"],
            s=55,
            alpha=0.75,
            label="Unmeasured"
        )

        if row_idx == 0:
            ax.set_title(title, fontweight="bold")

        ax.set_xlabel(
            f"PC1 ({pc1_var:.1f}%)",
            fontweight="bold"
        )

        ax.set_ylabel(
            f"PC2 ({pc2_var:.1f}%)",
            fontweight="bold"
        )

        ax.grid(True, alpha=0.3)

        if col_idx == 0:
            ax.text(
                -0.18,
                0.5,
                restriction,
                transform=ax.transAxes,
                fontsize=32,
                fontweight="bold",
                rotation=90,
                va="center",
                ha="center"
            )

# ------------------------------------------------------------
# Shared legend
# ------------------------------------------------------------

handles, labels = axes[0, 0].get_legend_handles_labels()

legend = fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    fontsize=LEGEND_FONT_SIZE,
    frameon=True
)

# Set legend marker sizes explicitly
for handle in legend.legend_handles:
    handle.set_sizes([LEGEND_MARKER_SIZE**2])

plt.tight_layout(rect=[0, 0, 1, 0.94])


plt.show()

In [ ]:
# ============================================================
# Harmonized AOA comparison figure by restriction class
#
# Rows    = restriction classes
# Columns = topographic variables, bioclimatic variables,
#           and AEF embeddings
# ============================================================

import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# ============================================================
# 1. Global manuscript style
# ============================================================

# Font settings
FONT_FAMILY = "sans-serif"

BASE_FONT_SIZE = 12
TITLE_FONT_SIZE = 15
AXIS_LABEL_SIZE = 13
TICK_LABEL_SIZE = 11
ROW_LABEL_SIZE = 14
PANEL_LABEL_SIZE = 13
LEGEND_FONT_SIZE = 12

# Line settings
FRAME_LINE_WIDTH = 1.25
TICK_LINE_WIDTH = 1.10
GRID_LINE_WIDTH = 0.70

# Point settings
MEASURED_POINT_SIZE = 20
UNMEASURED_POINT_SIZE = 28

MEASURED_ALPHA = 0.45
UNMEASURED_ALPHA = 0.80

# Consistent, colorblind-friendly colors
MEASURED_COLOR = "#595959"
UNMEASURED_COLOR = "#D55E00"

# Background and grid colors
BACKGROUND_COLOR = "white"
GRID_COLOR = "#D9D9D9"
FRAME_COLOR = "black"

# Figure dimensions
FIGURE_WIDTH_PER_COLUMN = 5.8
FIGURE_HEIGHT_PER_ROW = 4.6

# Optional panel labels: (a), (b), (c), ...
ADD_PANEL_LABELS = False # Modified: Set to False to remove panel labels

# Export settings
OUTPUT_FILE = "AOA_PCA_restrictions_harmonized.png"
OUTPUT_DPI = 600


plt.rcParams.update({
    # Fonts
    "font.family": FONT_FAMILY,
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": BASE_FONT_SIZE,

    "axes.titlesize": TITLE_FONT_SIZE,
    "axes.titleweight": "semibold",
    "axes.labelsize": AXIS_LABEL_SIZE,
    "axes.labelweight": "semibold",

    "xtick.labelsize": TICK_LABEL_SIZE,
    "ytick.labelsize": TICK_LABEL_SIZE,

    "legend.fontsize": LEGEND_FONT_SIZE,

    # Backgrounds
    "figure.facecolor": BACKGROUND_COLOR,
    "axes.facecolor": BACKGROUND_COLOR,
    "savefig.facecolor": BACKGROUND_COLOR,

    # Axes and frames
    "axes.edgecolor": FRAME_COLOR,
    "axes.linewidth": FRAME_LINE_WIDTH,
    "axes.axisbelow": True,

    # Gridlines
    "axes.grid": False, # Modified: Set to False to remove grid
    "grid.color": GRID_COLOR,
    "grid.linestyle": "-",
    "grid.linewidth": GRID_LINE_WIDTH,
    "grid.alpha": 0.75,

    # Tick marks
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": TICK_LINE_WIDTH,
    "ytick.major.width": TICK_LINE_WIDTH,
    "xtick.major.size": 5,
    "ytick.major.size": 5,

    # Legend
    "legend.frameon": True,
    "legend.fancybox": False,
    "legend.framealpha": 1.0,

    # Editable fonts in vector outputs
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


# ============================================================
# 2. Reusable axis-formatting function
# ============================================================

def apply_manuscript_axis_style(ax):
    """
    Apply the same background, grid, frame, and tick style
    to every subplot.
    """

    ax.set_facecolor(BACKGROUND_COLOR)
    ax.set_axisbelow(True)

    # Use major gridlines only
    # Modified: Removed ax.grid call to ensure no grid is shown
    # ax.grid(
    #     visible=True,
    #     which="major",
    #     axis="both",
    #     color=GRID_COLOR,
    #     linestyle="-",
    #     linewidth=GRID_LINE_WIDTH,
    #     alpha=0.75
    # )

    # Remove minor gridlines and minor ticks
    ax.grid(False, which="minor") # Keep for consistency even if major grid is off
    ax.minorticks_off()

    # Show a complete rectangular frame
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(FRAME_COLOR)
        spine.set_linewidth(FRAME_LINE_WIDTH)

    # Standardize ticks
    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=5,
        width=TICK_LINE_WIDTH,
        color=FRAME_COLOR,
        labelcolor="black",
        top=False,
        right=False
    )


# ============================================================
# 3. PCA helper
# ============================================================

def get_pca_scores_by_restriction(data, features):
    """
    Standardize predictor variables using measured plots,
    fit PCA using measured plots, and project both measured
    and unmeasured plots into the same PCA space.
    """

    required_columns = ["plot_status", "Restrictions"] + features

    missing_columns = [
        column for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            "The following required columns are missing:\n"
            + ", ".join(missing_columns)
        )

    sub = data[required_columns].copy()

    # Convert infinite values to missing values
    sub = sub.replace([np.inf, -np.inf], np.nan)

    # Retain complete observations for the selected feature set
    sub = sub.dropna(subset=features)

    measured = sub[
        sub["plot_status"].str.lower() == "measured"
    ].copy()

    unmeasured = sub[
        sub["plot_status"].str.lower() == "unmeasured"
    ].copy()

    if measured.empty:
        raise ValueError(
            "No complete measured observations were available "
            "for the selected feature set."
        )

    if unmeasured.empty:
        raise ValueError(
            "No complete unmeasured observations were available "
            "for the selected feature set."
        )

    scaler = StandardScaler()

    # Fit scaling only on measured plots
    measured_scaled = scaler.fit_transform(measured[features])

    # Apply the measured-plot scaling to unmeasured plots
    unmeasured_scaled = scaler.transform(unmeasured[features])

    pca = PCA(n_components=2)

    # Fit PCA only on measured plots
    measured_pca = pca.fit_transform(measured_scaled)

    # Project unmeasured plots into the measured PCA space
    unmeasured_pca = pca.transform(unmeasured_scaled)

    measured_scores = measured[
        ["plot_status", "Restrictions"]
    ].copy()

    measured_scores["PC1"] = measured_pca[:, 0]
    measured_scores["PC2"] = measured_pca[:, 1]

    unmeasured_scores = unmeasured[
        ["plot_status", "Restrictions"]
    ].copy()

    unmeasured_scores["PC1"] = unmeasured_pca[:, 0]
    unmeasured_scores["PC2"] = unmeasured_pca[:, 1]

    scores = pd.concat(
        [measured_scores, unmeasured_scores],
        axis=0,
        ignore_index=True
    )

    return scores, pca


# ============================================================
# 4. Axis-limit helper
# ============================================================

def calculate_axis_limits(scores, padding_fraction=0.05):
    """
    Calculate common PC1 and PC2 limits for all restriction
    classes within one predictor-space column.
    """

    x_min = scores["PC1"].min()
    x_max = scores["PC1"].max()

    y_min = scores["PC2"].min()
    y_max = scores["PC2"].max()

    x_range = x_max - x_min
    y_range = y_max - y_min

    # Prevent zero-width limits
    if x_range == 0:
        x_range = 1

    if y_range == 0:
        y_range = 1

    x_padding = x_range * padding_fraction
    y_padding = y_range * padding_fraction

    return (
        (x_min - x_padding, x_max + x_padding),
        (y_min - y_padding, y_max + y_padding)
    )


# ============================================================
# 5. Predictor groups
# ============================================================

topo_features = [
    "elevation",
    "slope",
    "aspect",
    "northness",
    "eastness",
    "tpi",
    "curvature"
]

bio_features = [
    "bio01", "bio02", "bio03", "bio04", "bio05",
    "bio06", "bio07", "bio08", "bio09", "bio10",
    "bio11", "bio12", "bio13", "bio14", "bio15",
    "bio16", "bio17", "bio18", "bio19"
]

# Identify AEF dimensions such as A00–A63
AEF_COLS = [
    column for column in df.columns
    if (
        isinstance(column, str)
        and len(column) == 3
        and column.startswith("A")
        and column[1:].isdigit()
    )
]

AEF_COLS = sorted(
    AEF_COLS,
    key=lambda column: int(column[1:])
)

if not AEF_COLS:
    raise ValueError(
        "No AEF columns were found. Expected names such as "
        "A00, A01, ..., A63."
    )


# ============================================================
# 6. Calculate PCA spaces
# ============================================================

topo_scores, topo_pca = get_pca_scores_by_restriction(
    df,
    topo_features
)

bio_scores, bio_pca = get_pca_scores_by_restriction(
    df,
    bio_features
)

aef_scores, aef_pca = get_pca_scores_by_restriction(
    df,
    AEF_COLS
)

datasets = [
    {
        "scores": topo_scores,
        "pca": topo_pca,
        "title": "Topographic variables"
    },
    {
        "scores": bio_scores,
        "pca": bio_pca,
        "title": "Bioclimatic variables"
    },
    {
        "scores": aef_scores,
        "pca": aef_pca,
        "title": "AEF embeddings"
    }
]


# ============================================================
# 7. Restriction-class order
# ============================================================

available_restrictions = (
    df.loc[
        df["plot_status"].str.lower() == "unmeasured",
        "Restrictions"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if not available_restrictions:
    raise ValueError(
        "No restriction classes were found among unmeasured plots."
    )

# Preferred manuscript order
preferred_order = ["HCA", "LRA", "RAA"]

restriction_classes = [
    restriction
    for restriction in preferred_order
    if restriction in available_restrictions
]

# Add any other restriction classes alphabetically
restriction_classes.extend(
    sorted(
        restriction
        for restriction in available_restrictions
        if restriction not in preferred_order
    )
)

n_rows = len(restriction_classes)
n_cols = len(datasets)


# ============================================================
# 8. Use common axis limits within each PCA column
# ============================================================

column_limits = []

for dataset in datasets:

    x_limits, y_limits = calculate_axis_limits(
        dataset["scores"],
        padding_fraction=0.05
    )

    column_limits.append({
        "x": x_limits,
        "y": y_limits
    })


# ============================================================
# 9. Create figure
# ============================================================

fig, axes = plt.subplots(
    nrows=n_rows,
    ncols=n_cols,
    figsize=(
        FIGURE_WIDTH_PER_COLUMN * n_cols,
        FIGURE_HEIGHT_PER_ROW * n_rows
    ),
    squeeze=False,
    facecolor=BACKGROUND_COLOR
)

panel_letters = list(string.ascii_lowercase)


# ============================================================
# 10. Plot the PCA comparisons
# ============================================================

for row_idx, restriction in enumerate(restriction_classes):

    for col_idx, dataset in enumerate(datasets):

        ax = axes[row_idx, col_idx]

        scores = dataset["scores"]
        pca = dataset["pca"]
        title = dataset["title"]

        measured = scores[
            scores["plot_status"].str.lower() == "measured"
        ]

        unmeasured_restriction = scores[
            (
                scores["plot_status"].str.lower()
                == "unmeasured"
            )
            &
            (
                scores["Restrictions"].astype(str)
                == restriction
            )
        ]

        pc1_variance = (
            pca.explained_variance_ratio_[0] * 100
        )

        pc2_variance = (
            pca.explained_variance_ratio_[1] * 100
        )

        # ----------------------------------------------------
        # Measured plots
        # ----------------------------------------------------

        ax.scatter(
            measured["PC1"],
            measured["PC2"],
            s=MEASURED_POINT_SIZE,
            c=MEASURED_COLOR,
            alpha=MEASURED_ALPHA,
            marker="o",
            edgecolors="none",
            linewidths=0,
            label="Measured",
            rasterized=True,
            zorder=2
        )

        # ----------------------------------------------------
        # Unmeasured plots from the selected restriction class
        # ----------------------------------------------------

        ax.scatter(
            unmeasured_restriction["PC1"],
            unmeasured_restriction["PC2"],
            s=UNMEASURED_POINT_SIZE,
            c=UNMEASURED_COLOR,
            alpha=UNMEASURED_ALPHA,
            marker="o",
            edgecolors="none",
            linewidths=0,
            label="Unmeasured",
            rasterized=True,
            zorder=3
        )

        # Column titles shown only in the first row
        if row_idx == 0:
            ax.set_title(
                title,
                pad=12
            )

        ax.set_xlabel(
            f"PC1 ({pc1_variance:.1f}%)",
            labelpad=7
        )

        ax.set_ylabel(
            f"PC2 ({pc2_variance:.1f}%)",
            labelpad=7
        )

        # Identical limits across rows within each column
        ax.set_xlim(column_limits[col_idx]["x"])
        ax.set_ylim(column_limits[col_idx]["y"])

        # Apply standardized frame and grid style
        apply_manuscript_axis_style(ax)

        # ----------------------------------------------------
        # Row labels
        # ----------------------------------------------------

        if col_idx == 0:
            ax.text(
                -0.23,
                0.50,
                restriction,
                transform=ax.transAxes,
                fontsize=ROW_LABEL_SIZE,
                fontweight="semibold",
                rotation=90,
                va="center",
                ha="center",
                color="black"
            )

        # ----------------------------------------------------
        # Panel labels
        # ----------------------------------------------------

        panel_number = row_idx * n_cols + col_idx

        if ADD_PANEL_LABELS and panel_number < len(panel_letters):
            ax.text(
                0.025,
                0.975,
                f"({panel_letters[panel_number]})",
                transform=ax.transAxes,
                fontsize=PANEL_LABEL_SIZE,
                fontweight="bold",
                va="top",
                ha="left",
                color="black",
                zorder=5
            )


# ============================================================
# 11. Shared legend
# ============================================================

handles, labels = axes[0, 0].get_legend_handles_labels()

legend = fig.legend(
    handles=handles,
    labels=labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    ncol=2,
    markerscale=1.7,
    columnspacing=2.2,
    handletextpad=0.7,
    borderpad=0.7,
    frameon=True,
    fancybox=False
)

# Modified: Bold the legend text
for text in legend.get_texts():
    text.set_fontweight('bold')

legend_frame = legend.get_frame()
legend_frame.set_facecolor(BACKGROUND_COLOR)
legend_frame.set_edgecolor(FRAME_COLOR)
legend_frame.set_linewidth(FRAME_LINE_WIDTH)
legend_frame.set_alpha(1.0)


# ============================================================
# 12. Layout and export
# ============================================================

fig.tight_layout(
    rect=[0.06, 0.03, 0.99, 0.94],
    h_pad=1.5,
    w_pad=1.5
)

fig.savefig(
    OUTPUT_FILE,
    dpi=OUTPUT_DPI,
    bbox_inches="tight",
    facecolor=BACKGROUND_COLOR
)



plt.show()

# UMAP Visualization

In [ ]:
import umap
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from IPython.display import Image # Import Image for displaying local files

# =========================================================
# DATA
# =========================================================

df_umap_source = merged_embeddings_field_dem_bioclim.copy()

# =========================================================
# AEF EMBEDDINGS
# =========================================================

def _sort_A(cols):
    return sorted(
        cols,
        key=lambda x: int(x[1:]) if isinstance(x, str) and x[1:].isdigit() else 10**9
    )

AEF_COLS = [
    c for c in df_umap_source.columns
    if len(c) == 3 and c[0] == "A" and c[1:].isdigit()
]

AEF_COLS = _sort_A(AEF_COLS)

# =========================================================
# TARGET VARIABLES
# =========================================================

TARGETS_UMAP = [
    "aspect",
    "eastness",
    "elevation",
    "northness",
    "slope"
]

TARGET_LABELS = {
    "aspect": "Aspect",
    "eastness": "Eastness",
    "elevation": "Elevation",
    "northness": "Northness",
    "slope": "Slope"
}

COLORBAR_LABELS = {
    "aspect": "Aspect",
    "eastness": "Eastness",
    "elevation": "Elevation (m)",
    "northness": "Northness",
    "slope": "Slope (°)"
}

# =========================================================
# UMAP SETTINGS
# =========================================================

UMAP_PARAMS = dict(
    n_neighbors=35,
    min_dist=0.05,
    n_components=2,
    random_state=42,
)

UMAP_METRIC = "manhattan"

POINT_SIZE = 8
POINT_ALPHA = 0.75
CMAP = "coolwarm"
# USE_CUSTOM_LIMITS = True # Commented out to allow dynamic axis limits

# =========================================================
# FIGURE STYLE
# =========================================================

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
    "font.family": "sans-serif",
    "font.size": 20,
    "axes.titlesize": 23,
    "axes.labelsize": 21,
    "xtick.labelsize": 19,
    "ytick.labelsize": 19,
})

# =========================================================
# CORE FUNCTIONS
# =========================================================

def run_umap(df, feature_cols):
    X = df[feature_cols].copy()
    X = SimpleImputer(strategy="median").fit_transform(X)
    X = StandardScaler().fit_transform(X)

    reducer = umap.UMAP(
        metric=UMAP_METRIC,
        **UMAP_PARAMS
    )

    embedding = reducer.fit_transform(X)
    return embedding

def plot_empty_panel(ax, reason):
    ax.set_facecolor("#f7f8fc")
    ax.text(
        0.5,
        0.5,
        reason,
        ha="center",
        va="center",
        transform=ax.transAxes,
        color="#9a9dac",
        fontsize=14
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("") # Clear any default title

def plot_umap_panel(ax, embedding, target_vals, target_col):

    vmin, vmax = np.nanpercentile(target_vals, [2, 98])
    norm = Normalize(vmin=vmin, vmax=vmax)

    ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=target_vals,
        cmap=CMAP,
        norm=norm,
        s=POINT_SIZE,
        alpha=POINT_ALPHA,
        linewidths=0,
        rasterized=True
    )

    ax.set_title(TARGET_LABELS.get(target_col, target_col), fontweight="bold", pad=6) # Changed title to reflect target

    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

    # Removed custom xlim and ylim to allow automatic scaling
    # if USE_CUSTOM_LIMITS:
    #     ax.set_xlim(5, 15)
    #     ax.set_ylim(0, 10)

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("black")

    cbar = plt.colorbar(
        ScalarMappable(norm=norm, cmap=CMAP),
        ax=ax,
        fraction=0.035,
        pad=0.015
    )

    cbar_label = COLORBAR_LABELS.get(
        target_col,
        TARGET_LABELS.get(target_col, target_col)
    )

    cbar.set_label(cbar_label, fontsize=16)
    cbar.ax.tick_params(labelsize=15)
    cbar.outline.set_linewidth(0.8)

# =========================================================
# PREPARE DATA ONCE
# =========================================================

required_cols = AEF_COLS + TARGETS_UMAP
existing_targets = [c for c in TARGETS_UMAP if c in df_umap_source.columns]

df_umapper = df_umap_source.dropna(
    subset=AEF_COLS + existing_targets,
    how="any"
).copy()

print(f"Number of samples used: {len(df_umapper)}")
print(f"Number of AEF features: {len(AEF_COLS)}")

embedding = run_umap(df_umapper, AEF_COLS)

# =========================================================
# GENERATE SINGLE FIGURE WITH 2 COLUMNS AND 4 ROWS
# =========================================================

n_rows = 2
n_cols = 3
# Adjust figsize for 4 rows and 2 columns
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7.5 * n_cols, 10.8 * n_rows), squeeze=False)
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration



for idx, target in enumerate(TARGETS_UMAP):
    ax = axes[idx] # Get the current subplot axis

    if target not in df_umapper.columns:
        plot_empty_panel(ax, f"{target}\nnot found")
        continue

    target_vals = df_umapper[target].values

    plot_umap_panel(
        ax=ax,
        embedding=embedding,
        target_vals=target_vals,
        target_col=target
    )

# Fill any remaining subplots with empty panels
for idx_empty in range(len(TARGETS_UMAP), n_rows * n_cols):
    ax = axes[idx_empty]
    plot_empty_panel(ax, "")

plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout to make space for suptitle

out_name = "UMAP_AEF_structural_variables.png" # Define out_name here

plt.show()


# Explicitly display the saved image file

print("\nFinished: combined UMAP figure saved.")

In [ ]:
import umap
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import math

# =========================================================
# DATA
# =========================================================

df_umap_source = merged_embeddings_field_dem_bioclim.copy()

# =========================================================
# AEF EMBEDDINGS
# =========================================================

def _sort_A(cols):
    return sorted(
        cols,
        key=lambda x: int(x[1:]) if isinstance(x, str) and x[1:].isdigit() else 10**9
    )

AEF_COLS = [
    c for c in df_umap_source.columns
    if len(c) == 3 and c[0] == "A" and c[1:].isdigit()
]

AEF_COLS = _sort_A(AEF_COLS)

# =========================================================
# BIOCLIM TARGET VARIABLES
# =========================================================

TARGETS_UMAP = [
    "bio01", "bio02", "bio04",
    "bio05", "bio06", "bio07", "bio08",
    "bio09", "bio10", "bio11", "bio12",
    "bio13", "bio14", "bio15", "bio16",
    "bio17", "bio18", "bio19"
]

TARGET_LABELS = {
    "bio01": "BIO1",
    "bio02": "BIO2",
    "bio04": "BIO4",
    "bio05": "BIO5",
    "bio06": "BIO6",
    "bio07": "BIO7",
    "bio08": "BIO8",
    "bio09": "BIO9",
    "bio10": "BIO10",
    "bio11": "BIO11",
    "bio12": "BIO12",
    "bio13": "BIO13",
    "bio14": "BIO14",
    "bio15": "BIO15",
    "bio16": "BIO16",
    "bio17": "BIO17",
    "bio18": "BIO18",
    "bio19": "BIO19"
}

COLORBAR_LABELS = {
    "bio01": "BIO1",
    "bio02": "BIO2",
    "bio04": "BIO4",
    "bio05": "BIO5",
    "bio06": "BIO6",
    "bio07": "BIO7",
    "bio08": "BIO8",
    "bio09": "BIO9",
    "bio10": "BIO10",
    "bio11": "BIO11",
    "bio12": "BIO12",
    "bio13": "BIO13",
    "bio14": "BIO14",
    "bio15": "BIO15",
    "bio16": "BIO16",
    "bio17": "BIO17",
    "bio18": "BIO18",
    "bio19": "BIO19"
}

# =========================================================
# UMAP SETTINGS
# =========================================================

UMAP_PARAMS = dict(
    n_neighbors=35,
    min_dist=0.05,
    n_components=2,
    random_state=42,
)

UMAP_METRIC = "manhattan"

POINT_SIZE = 8
POINT_ALPHA = 0.75
CMAP = "coolwarm"

USE_CUSTOM_LIMITS = True

XMIN = 5
XMAX = 15
YMIN = 2
YMAX = 12

# =========================================================
# FIGURE STYLE
# =========================================================

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
    "font.family": "sans-serif",
    "font.size": 20,
    "axes.titlesize": 23,
    "axes.labelsize": 21,
    "xtick.labelsize": 19,
    "ytick.labelsize": 19,
})

# =========================================================
# CORE FUNCTIONS
# =========================================================

def run_umap(df, feature_cols):
    X = df[feature_cols].copy()
    X = SimpleImputer(strategy="median").fit_transform(X)
    X = StandardScaler().fit_transform(X)

    reducer = umap.UMAP(
        metric=UMAP_METRIC,
        **UMAP_PARAMS
    )

    embedding = reducer.fit_transform(X)
    return embedding


def plot_empty_panel(ax, reason):
    ax.set_facecolor("white")
    ax.text(
        0.5,
        0.5,
        reason,
        ha="center",
        va="center",
        transform=ax.transAxes,
        color="black",
        fontsize=14
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("")


def plot_umap_panel(ax, embedding, target_vals, target_col):

    vmin, vmax = np.nanpercentile(target_vals, [2, 98])
    norm = Normalize(vmin=vmin, vmax=vmax)

    ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=target_vals,
        cmap=CMAP,
        norm=norm,
        s=POINT_SIZE,
        alpha=POINT_ALPHA,
        linewidths=0,
        rasterized=True
    )

    ax.set_title(
        TARGET_LABELS.get(target_col, target_col),
        fontweight="bold",
        pad=6
    )

    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

    if USE_CUSTOM_LIMITS:
        ax.set_xlim(XMIN, XMAX)
        ax.set_ylim(YMIN, YMAX)

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("black")

    cbar = plt.colorbar(
        ScalarMappable(norm=norm, cmap=CMAP),
        ax=ax,
        fraction=0.035,
        pad=0.015
    )

    cbar_label = COLORBAR_LABELS.get(
        target_col,
        TARGET_LABELS.get(target_col, target_col)
    )

    cbar.set_label(cbar_label, fontsize=16)
    cbar.ax.tick_params(labelsize=15)
    cbar.outline.set_linewidth(0.8)

# =========================================================
# PREPARE DATA ONCE
# =========================================================

existing_targets = [
    c for c in TARGETS_UMAP
    if c in df_umap_source.columns
]

df_umapper = df_umap_source.dropna(
    subset=AEF_COLS + existing_targets,
    how="any"
).copy()

print(f"Number of samples used: {len(df_umapper)}")
print(f"Number of AEF features: {len(AEF_COLS)}")
print(f"Number of bioclim variables found: {len(existing_targets)}")

embedding = run_umap(df_umapper, AEF_COLS)

# =========================================================
# GENERATE SEPARATE 2 x 4 IMAGES
# Each image contains 8 bioclim variables
# =========================================================

n_rows = 3
n_cols = 3
panels_per_image = n_rows * n_cols

n_images = math.ceil(len(TARGETS_UMAP) / panels_per_image)

for img_idx in range(n_images):

    start_idx = img_idx * panels_per_image
    end_idx = start_idx + panels_per_image

    current_targets = TARGETS_UMAP[start_idx:end_idx]

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(7.5 * n_cols, 10.8 * n_rows),
        squeeze=False
    )

    axes_flat = axes.flatten()

    for idx, target in enumerate(current_targets):

        ax = axes_flat[idx]

        if target not in df_umapper.columns:
            plot_empty_panel(ax, f"{target}\nnot found")
            continue

        target_vals = df_umapper[target].values

        plot_umap_panel(
            ax=ax,
            embedding=embedding,
            target_vals=target_vals,
            target_col=target
        )

    for idx_empty in range(len(current_targets), panels_per_image):
        ax = axes_flat[idx_empty]
        plot_empty_panel(ax, "No data")

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    out_name = f"UMAP_AEF_bioclim_part_{img_idx + 1}.png"


    plt.show()

    print(f"Saved: {out_name}")

print("\nFinished: separate 2 x 4 UMAP bioclim figures saved.")

# Random Forest

### Four-management-region spatial cross-validation

Model assessment uses **four-fold leave-one-management-region-out spatial cross-validation**. The four folds are **Gilan, Nowshahr, Sari, and Golestan**. In each iteration, one complete management region is withheld for validation and the remaining three regions are used for training.

**Before running this section:** set `MANAGEMENT_REGION_COL` to the actual dataframe column containing these management-region labels. The notebook deliberately does not infer regions from plot IDs or coordinates, because the uploaded code does not contain a verified region field/mapping.


In [ ]:
import gc
import warnings
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

# =========================================================
# INPUTS
# =========================================================

df_base = merged_embeddings_field_dem_bioclim.copy()

TARGETS_WANTED = [
    "Volume",   # standing volume
    "n_trees",  # tree density
    "max_h"     # canopy height
]

OUTLIER_SD_THRESH = 3   # residual SD threshold for outlier removal

# =========================================================
# HELPERS
# =========================================================

def _sort_A(cols):
    return sorted(
        cols,
        key=lambda x: int(x[1:]) if isinstance(x, str) and x[1:].isdigit() else 10**9
    )

# =========================================================
# FEATURES: AEF ONLY
# =========================================================

AEF_COLS = _sort_A([
    c for c in df_base.columns
    if len(c) == 3 and c[0] == "A" and c[1:].isdigit()
])

print(f"AEF features: {len(AEF_COLS)}")
print(f"Targets:      {TARGETS_WANTED}")

# =========================================================
# CROSS-VALIDATED RF — one pass
# =========================================================

# =========================================================
# FOUR-MANAGEMENT-REGION SPATIAL CROSS-VALIDATION
# =========================================================
# IMPORTANT:
# Set MANAGEMENT_REGION_COL to the column in your dataframe that identifies
# the four official management divisions. The values must resolve to:
# Gilan, Nowshahr, Sari, and Golestan.
MANAGEMENT_REGION_COL = "management_region"

EXPECTED_REGIONS = ["Gilan", "Nowshahr", "Sari", "Golestan"]

# Optional aliases make capitalization / common spellings robust.
REGION_ALIASES = {
    "gilan": "Gilan",
    "guilan": "Gilan",
    "nowshahr": "Nowshahr",
    "noshahr": "Nowshahr",
    "sari": "Sari",
    "golestan": "Golestan",
}

def _standardize_management_regions(series):
    raw = series.astype(str).str.strip()
    standardized = raw.str.lower().map(REGION_ALIASES)
    # Keep an exact expected name if it was already supplied.
    standardized = standardized.fillna(
        raw.where(raw.isin(EXPECTED_REGIONS))
    )
    return standardized

def _prepare_region_column(df):
    if MANAGEMENT_REGION_COL not in df.columns:
        raise KeyError(
            f"Column '{MANAGEMENT_REGION_COL}' was not found. "
            "Set MANAGEMENT_REGION_COL to the dataframe column containing "
            "Gilan, Nowshahr, Sari, and Golestan before running the models."
        )

    out = df.copy()
    out["_cv_region"] = _standardize_management_regions(
        out[MANAGEMENT_REGION_COL]
    )

    unresolved = out.loc[out["_cv_region"].isna(), MANAGEMENT_REGION_COL].dropna().unique()
    if len(unresolved):
        raise ValueError(
            "Unrecognized management-region values: "
            f"{list(unresolved)}. Update REGION_ALIASES if needed."
        )

    present = set(out["_cv_region"].dropna().unique())
    missing = set(EXPECTED_REGIONS) - present
    if missing:
        raise ValueError(
            f"Missing management region(s): {sorted(missing)}. "
            "All four regions are required for four-fold spatial CV."
        )
    return out

def run_cv(df, target, features, label=""):
    """
    Four-fold leave-one-management-region-out spatial cross-validation.

    Each fold withholds one complete management region (Gilan, Nowshahr,
    Sari, or Golestan) for validation and trains on the other three.
    """
    df_filtered = df[df[target].notna()].copy()
    df_filtered = _prepare_region_column(df_filtered)

    X = df_filtered[features]
    y = df_filtered[target]

    fold_results = []
    residuals = pd.Series(np.nan, index=df_filtered.index, dtype=float)
    y_true_oof = pd.Series(np.nan, index=df_filtered.index, dtype=float)
    y_pred_oof = pd.Series(np.nan, index=df_filtered.index, dtype=float)

    for test_region in EXPECTED_REGIONS:
        test_mask = df_filtered["_cv_region"].eq(test_region)
        train_mask = ~test_mask

        if test_mask.sum() == 0 or train_mask.sum() == 0:
            raise ValueError(f"Invalid fold for region: {test_region}")

        X_train = X.loc[train_mask]
        X_test = X.loc[test_mask]
        y_train = y.loc[train_mask]
        y_test = y.loc[test_mask]

        # Imputation is fitted only on the training regions to avoid leakage.
        imp = SimpleImputer(strategy="median")
        X_train_imp = imp.fit_transform(X_train)
        X_test_imp = imp.transform(X_test)

        model = RandomForestRegressor(
            n_estimators=500,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train_imp, y_train)
        y_pred = model.predict(X_test_imp)

        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(np.mean((y_test.to_numpy() - y_pred) ** 2))
        mae = mean_absolute_error(y_test, y_pred)
        fold_results.append([r2, rmse, mae])

        test_index = df_filtered.index[test_mask]
        residuals.loc[test_index] = np.abs(y_test.to_numpy() - y_pred)
        y_true_oof.loc[test_index] = y_test.to_numpy()
        y_pred_oof.loc[test_index] = y_pred

        print(
            f"    Held-out region: {test_region:<8} "
            f"(n={test_mask.sum():4d}) | "
            f"R²={r2:.3f}, RMSE={rmse:.3f}, MAE={mae:.3f}"
        )

    return (
        np.asarray(fold_results),
        residuals,
        df_filtered.drop(columns="_cv_region"),
        y_true_oof,
        y_pred_oof,
    )

# =========================================================
# MAIN LOOP
# =========================================================

all_metrics = []
preds_store_rf = {}

for target in TARGETS_WANTED:
    print(f"\nTarget: {target}")

    # Pass 1
    fold_arr_1, residuals, df_t, _, _ = run_cv(df_base, target, AEF_COLS)
    all_metrics.append({
        "Target": target, "Pass": "Original", "n": len(df_t), "n_removed": 0,
        "R2_mean": fold_arr_1[:, 0].mean(), "R2_sd": fold_arr_1[:, 0].std(),
        "RMSE_mean": fold_arr_1[:, 1].mean(), "RMSE_sd": fold_arr_1[:, 1].std(),
        "MAE_mean": fold_arr_1[:, 2].mean(), "MAE_sd": fold_arr_1[:, 2].std(),
    })

    # Outlier removal
    threshold = residuals.mean() + OUTLIER_SD_THRESH * residuals.std()
    df_clean = df_t.loc[residuals <= threshold]
    n_removed = len(df_t) - len(df_clean)

    # Pass 2
    fold_arr_2, _, _, y_true_clean, y_pred_clean = run_cv(df_clean, target, AEF_COLS)
    all_metrics.append({
        "Target": target, "Pass": "Outliers removed", "n": len(df_clean), "n_removed": n_removed,
        "R2_mean": fold_arr_2[:, 0].mean(), "R2_sd": fold_arr_2[:, 0].std(),
        "RMSE_mean": fold_arr_2[:, 1].mean(), "RMSE_sd": fold_arr_2[:, 1].std(),
        "MAE_mean": fold_arr_2[:, 2].mean(), "MAE_sd": fold_arr_2[:, 2].std(),
    })

    preds_store_rf[target] = {
        "AEF embeddings": {"y_true": y_true_clean, "y_pred": y_pred_clean}
    }
    print(f"  Pass 2 (cleaned, n={len(df_clean)}) R²={fold_arr_2[:,0].mean():.3f}")

results_df = pd.DataFrame(all_metrics)

In [ ]:
import pandas as pd
import numpy as np

# Assuming preds_store_rf is available from the previous cell (HBNeD6ZEizwn)
# and TARGETS is available from cell izsaHehoL4TS
# Re-defining TARGETS_FOR_RESIDUAL_PLOTS for explicit scope within this cell
TARGETS_FOR_RESIDUAL_PLOTS = ["Volume", "n_trees", "max_h"]

# Construct residual_df_for_plotting from preds_store_rf
residual_records = []

for target in TARGETS_FOR_RESIDUAL_PLOTS:
    # Assuming only 'AEF embeddings' is used, as per the previous context and plots
    bandset_name = "AEF embeddings"

    # Check if data exists for the target and bandset
    if target in preds_store_rf and bandset_name in preds_store_rf[target]:
        pred_data = preds_store_rf[target][bandset_name]
        y_true = pred_data['y_true']
        y_pred = pred_data['y_pred']

        # Calculate absolute residuals
        abs_residual = (y_true - y_pred).abs()

        # Create a temporary DataFrame for this target/bandset combination
        temp_df = pd.DataFrame({
            'abs_residual': abs_residual,
            'target': target,
            'bandset': bandset_name,
            # Add the actual values of the target for binning
            f'actual_{target.lower()}': y_true # Column name like 'actual_volume'
        })
        residual_records.append(temp_df)
    else:
        print(f"Warning: No prediction data found for target '{target}' and bandset '{bandset_name}'.")

if residual_records:
    residual_df_for_plotting = pd.concat(residual_records, ignore_index=True)
    print("Prepared `residual_df_for_plotting` for residual analysis.")
    display(residual_df_for_plotting.head())
    print(residual_df_for_plotting.info())
else:
    print("No residual data could be prepared for plotting.")

In [ ]:
# Tidy display: round floats, keep n columns as int
display(
    results_df
    .assign(**{
        "R2 (Mean ± SD)": lambda d: d["R2_mean"].round(2).astype(str) + " ± " + d["R2_sd"].round(2).astype(str),
        "RMSE (Mean ± SD)": lambda d: d["RMSE_mean"].round(2).astype(str) + " ± " + d["RMSE_sd"].round(2).astype(str),
        "MAE (Mean ± SD)": lambda d: d["MAE_mean"].round(2).astype(str) + " ± " + d["MAE_sd"].round(2).astype(str),
    })
    [["Target", "Pass", "n", "n_removed", "R2 (Mean ± SD)", "RMSE (Mean ± SD)", "MAE (Mean ± SD)"]]
)

## Actual vs prediction hexabine

In [ ]:
# ============================================================
# Harmonized observed-versus-predicted density plots
#
# Columns = forest structural attributes
# Rows    = predictor representations
# ============================================================

import string
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.ticker import MaxNLocator
from scipy.stats import gaussian_kde
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


# ============================================================
# 1. Configuration
# ============================================================

TARGETS = [
    "Volume",
    "n_trees",
    "max_h"
]

BANDSETS = [
    "AEF embeddings"
]

TARGET_TITLES = {
    "Volume": "Standing volume",
    "n_trees": "Tree density",
    "max_h": "Canopy height"
}

TARGET_UNITS = {
    "Volume": "m³ ha⁻¹",
    "n_trees": "trees ha⁻¹",
    "max_h": "m"
}

# Plot appearance
FONT_FAMILY = "sans-serif"

BASE_FONT_SIZE = 12
TITLE_FONT_SIZE = 15
AXIS_LABEL_SIZE = 13
TICK_LABEL_SIZE = 11
PANEL_LABEL_SIZE = 13
COLORBAR_LABEL_SIZE = 11
COLORBAR_TICK_SIZE = 10
METRIC_FONT_SIZE = 10
ROW_LABEL_SIZE = 13

FRAME_LINE_WIDTH = 1.25
TICK_LINE_WIDTH = 1.10
GRID_LINE_WIDTH = 0.70

IDENTITY_LINE_WIDTH = 1.40
RUNNING_LINE_WIDTH = 1.80

BACKGROUND_COLOR = "white"
FRAME_COLOR = "black"
GRID_COLOR = "#D9D9D9"

IDENTITY_LINE_COLOR = "black"
RUNNING_LINE_COLOR = "#D55E00"

# Sequential and perceptually uniform density colormap
DENSITY_CMAP = "viridis"

# Panel dimensions
FIGURE_WIDTH_PER_PANEL = 5.3
FIGURE_HEIGHT_PER_PANEL = 5.0

# Optional metrics inside each panel
SHOW_METRICS = True

# Optional running median and interquartile range
SHOW_RUNNING_BAND = True

# Export settings
OUTPUT_PNG = "parity_density_AEF_harmonized.png"
OUTPUT_PDF = "parity_density_AEF_harmonized.pdf"
OUTPUT_DPI = 600


# ============================================================
# 2. Harmonized Matplotlib style
# ============================================================

MANUSCRIPT_STYLE = {
    # Fonts
    "font.family": FONT_FAMILY,
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": BASE_FONT_SIZE,

    "axes.titlesize": TITLE_FONT_SIZE,
    "axes.titleweight": "semibold",

    "axes.labelsize": AXIS_LABEL_SIZE,
    "axes.labelweight": "semibold",

    "xtick.labelsize": TICK_LABEL_SIZE,
    "ytick.labelsize": TICK_LABEL_SIZE,

    # Backgrounds
    "figure.facecolor": BACKGROUND_COLOR,
    "axes.facecolor": BACKGROUND_COLOR,
    "savefig.facecolor": BACKGROUND_COLOR,

    # Complete axis frames
    "axes.edgecolor": FRAME_COLOR,
    "axes.linewidth": FRAME_LINE_WIDTH,
    "axes.spines.top": True,
    "axes.spines.right": True,

    # Gridlines
    "axes.grid": False, # Modified: Set to False to remove grid
    "axes.axisbelow": True,
    "grid.color": GRID_COLOR,
    "grid.linestyle": "-",
    "grid.linewidth": GRID_LINE_WIDTH,
    "grid.alpha": 0.75,

    # Ticks
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": TICK_LINE_WIDTH,
    "ytick.major.width": TICK_LINE_WIDTH,
    "xtick.major.size": 5,
    "ytick.major.size": 5,

    # Editable fonts in vector outputs
    "pdf.fonttype": 42,
    "ps.fonttype": 42
}


# ============================================================
# 3. Reusable axis-formatting function
# ============================================================

def apply_manuscript_axis_style(ax):
    """
    Apply the same background, gridline, frame, and tick
    settings used in the other harmonized manuscript figures.
    """

    ax.set_facecolor(BACKGROUND_COLOR)
    ax.set_axisbelow(True)

    # Major gridlines only
    # ax.grid(
    #     visible=True,
    #     which="major",
    #     axis="both",
    #     color=GRID_COLOR,
    #     linestyle="-",
    #     linewidth=GRID_LINE_WIDTH,
    #     alpha=0.75
    # )

    ax.grid(False, which="minor")
    ax.minorticks_off()

    # Complete rectangular frame
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(FRAME_COLOR)
        spine.set_linewidth(FRAME_LINE_WIDTH)

    # Standardized tick marks
    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=5,
        width=TICK_LINE_WIDTH,
        color=FRAME_COLOR,
        labelcolor="black",
        top=False,
        right=False
    )


# ============================================================
# 4. Prediction-store helper
# ============================================================

def get_prediction_store(preds_store, target, bandset):
    """
    Retrieve predictions for one target–bandset combination.
    """

    return preds_store.get(target, {}).get(bandset)


def print_available_prediction_keys(
    preds_store,
    targets=TARGETS
):
    """
    Print the predictor-representation keys available for
    each response variable.
    """

    print("\nAvailable predictor keys in the prediction store:")

    for target in targets:

        target_store = preds_store.get(target, {})
        keys = sorted(target_store.keys())

        print(f"\n{target}:")

        if keys:
            for key in keys:
                print(f"  - {key}")
        else:
            print("  No prediction sets found")


# ============================================================
# 5. Performance metrics
# ============================================================

def calculate_metrics(y_true, y_pred):
    """
    Calculate R², RMSE, and MAE directly from the plotted
    observed and predicted values.
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    metrics = {
        "R2": np.nan,
        "RMSE": np.nan,
        "MAE": np.nan
    }

    if y_true.size == 0:
        return metrics

    metrics["RMSE"] = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    metrics["MAE"] = mean_absolute_error(
        y_true,
        y_pred
    )

    # R² requires at least two observations and nonzero
    # variance in the observed values
    if (
        y_true.size >= 2
        and not np.isclose(np.var(y_true), 0)
    ):
        metrics["R2"] = r2_score(
            y_true,
            y_pred
        )

    return metrics


def format_metric_value(value, decimals=2):
    """
    Safely format finite and missing metric values.
    """

    if np.isfinite(value):
        return f"{value:.{decimals}f}"

    return "NA"


# ============================================================
# 6. Running median and interquartile band
# ============================================================

def calculate_running_band(
    x,
    y,
    nbins=25,
    qlo=25,
    qhi=75
):
    """
    Calculate the median prediction and prediction
    interquartile range across quantile bins of observed
    values.
    """

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    x = x[valid]
    y = y[valid]

    if x.size == 0:
        return (
            np.array([]),
            np.array([]),
            np.array([]),
            np.array([])
        )

    # Quantile bins give approximately equal numbers of
    # observations per bin.
    edges = np.quantile(
        x,
        np.linspace(0, 1, nbins + 1)
    )

    # Remove duplicate edges caused by repeated observed values
    edges = np.unique(edges)

    if edges.size < 2:
        return (
            np.array([]),
            np.array([]),
            np.array([]),
            np.array([])
        )

    # Include extreme values in the first and final bins
    edges[0] -= 1e-9
    edges[-1] += 1e-9

    bin_index = np.digitize(
        x,
        edges,
        right=False
    ) - 1

    bin_index = np.clip(
        bin_index,
        0,
        len(edges) - 2
    )

    x_centers = []
    y_medians = []
    y_lower = []
    y_upper = []

    for bin_number in range(len(edges) - 1):

        selected = bin_index == bin_number

        if not np.any(selected):
            continue

        x_centers.append(
            np.median(x[selected])
        )

        y_medians.append(
            np.median(y[selected])
        )

        y_lower.append(
            np.percentile(y[selected], qlo)
        )

        y_upper.append(
            np.percentile(y[selected], qhi)
        )

    return (
        np.asarray(x_centers),
        np.asarray(y_medians),
        np.asarray(y_lower),
        np.asarray(y_upper)
    )


# ============================================================
# 7. Axis-limit helper
# ============================================================

def calculate_parity_limits(
    y_true,
    y_pred,
    padding_fraction=0.05
):
    """
    Calculate identical x- and y-axis limits so that the
    one-to-one line is correctly represented.
    """

    combined = np.concatenate([
        np.asarray(y_true, dtype=float),
        np.asarray(y_pred, dtype=float)
    ])

    combined = combined[np.isfinite(combined)]

    if combined.size == 0:
        return 0, 1

    lower = float(np.min(combined))
    upper = float(np.max(combined))

    value_range = upper - lower

    if np.isclose(value_range, 0):
        value_range = max(abs(lower), 1.0)

    padding = padding_fraction * value_range

    return lower - padding, upper + padding


# ============================================================
# 8. Colorbar-formatting function
# ============================================================

def format_colorbar(
    colorbar,
    label
):
    """
    Apply the same frame, tick, and label style to every
    density colorbar.
    """

    colorbar.set_label(
        label,
        fontsize=COLORBAR_LABEL_SIZE,
        fontweight="semibold",
        labelpad=8
    )

    colorbar.ax.tick_params(
        axis="y",
        which="major",
        direction="out",
        length=4,
        width=TICK_LINE_WIDTH,
        colors="black",
        labelsize=COLORBAR_TICK_SIZE
    )

    colorbar.outline.set_visible(True)
    colorbar.outline.set_edgecolor(FRAME_COLOR)
    colorbar.outline.set_linewidth(FRAME_LINE_WIDTH)


# ============================================================
# 9. Main plotting function
# ============================================================

def parity_density_grid(
    preds_store,
    bandsets=BANDSETS,
    targets=TARGETS,
    mode="hexbin",
    gridsize=45,
    bins=None,
    kde_resolution=200,
    show_running=SHOW_RUNNING_BAND,
    running_bins=25,
    interquartile_range=(25, 75),
    show_metrics=SHOW_METRICS
):
    """
    Generate harmonized observed-versus-predicted density
    plots for all requested targets and predictor sets.

    Parameters
    ----------
    mode : {"hexbin", "kde"}
        Density representation.

    bins : None or "log"
        Hexbin density scaling. Use None for raw observation
        counts and "log" for logarithmic count scaling.
    """

    n_rows = len(bandsets)
    n_cols = len(targets)

    with mpl.rc_context(MANUSCRIPT_STYLE):

        fig, axes = plt.subplots(
            nrows=n_rows,
            ncols=n_cols,
            figsize=(
                FIGURE_WIDTH_PER_PANEL * n_cols,
                FIGURE_HEIGHT_PER_PANEL * n_rows
            ),
            squeeze=False,
            facecolor=BACKGROUND_COLOR,
            constrained_layout=True
        )

        panel_letters = list(string.ascii_lowercase)

        for row_index, bandset in enumerate(bandsets):

            for column_index, target in enumerate(targets):

                ax = axes[row_index, column_index]

                panel_number = (
                    row_index * n_cols
                    + column_index
                )

                store = get_prediction_store(
                    preds_store,
                    target,
                    bandset
                )

                # --------------------------------------------
                # Missing predictions
                # --------------------------------------------

                if store is None:

                    apply_manuscript_axis_style(ax)

                    ax.text(
                        0.5,
                        0.5,
                        "Predictions unavailable",
                        transform=ax.transAxes,
                        ha="center",
                        va="center",
                        fontsize=BASE_FONT_SIZE,
                        color="#666666"
                    )

                    ax.set_xticks([])
                    ax.set_yticks([])

                    continue

                y_true = np.asarray(
                    store.get("y_true", [])
                ).ravel()

                y_pred = np.asarray(
                    store.get("y_pred", [])
                ).ravel()

                valid = (
                    np.isfinite(y_true)
                    & np.isfinite(y_pred)
                )

                y_true = y_true[valid]
                y_pred = y_pred[valid]

                if y_true.size == 0:

                    apply_manuscript_axis_style(ax)

                    ax.text(
                        0.5,
                        0.5,
                        "No valid predictions",
                        transform=ax.transAxes,
                        ha="center",
                        va="center",
                        fontsize=BASE_FONT_SIZE,
                        color="#666666"
                    )

                    ax.set_xticks([])
                    ax.set_yticks([])

                    continue

                lower_limit, upper_limit = (
                    calculate_parity_limits(
                        y_true,
                        y_pred,
                        padding_fraction=0.05
                    )
                )

                # --------------------------------------------
                # One-to-one reference line
                # --------------------------------------------

                ax.plot(
                    [lower_limit, upper_limit],
                    [lower_limit, upper_limit],
                    color=IDENTITY_LINE_COLOR,
                    linewidth=IDENTITY_LINE_WIDTH,
                    linestyle="--",
                    dashes=(5, 4),
                    label="1:1 line",
                    zorder=4
                )

                # --------------------------------------------
                # Hexagonal density
                # --------------------------------------------

                if mode.lower() == "hexbin":

                    density_artist = ax.hexbin(
                        y_true,
                        y_pred,
                        gridsize=gridsize,
                        bins=bins,
                        mincnt=1,
                        cmap=DENSITY_CMAP,
                        linewidths=0,
                        rasterized=True,
                        zorder=2
                    )

                    colorbar = fig.colorbar(
                        density_artist,
                        ax=ax,
                        fraction=0.046,
                        pad=0.035
                    )

                    if bins == "log":
                        density_label = (
                            "Observations per hexagon\n(log scale)"
                        )
                    else:
                        density_label = (
                            "Observations per hexagon"
                        )

                        # Raw hexbin counts should use integer ticks
                        colorbar.locator = MaxNLocator(
                            nbins=5,
                            integer=True
                        )

                        colorbar.update_ticks()

                    format_colorbar(
                        colorbar,
                        density_label
                    )

                # --------------------------------------------
                # Kernel-density representation
                # --------------------------------------------

                elif mode.lower() == "kde":

                    xx, yy = np.mgrid[
                        lower_limit:upper_limit:complex(
                            kde_resolution
                        ),
                        lower_limit:upper_limit:complex(
                            kde_resolution
                        )
                    ]

                    try:

                        kde = gaussian_kde(
                            np.vstack([
                                y_true,
                                y_pred
                            ])
                        )

                        density_values = kde(
                            np.vstack([
                                xx.ravel(),
                                yy.ravel()
                            ])
                        ).reshape(xx.shape)

                        density_artist = ax.imshow(
                            density_values.T,
                            origin="lower",
                            extent=[
                                lower_limit,
                                upper_limit,
                                lower_limit,
                                upper_limit
                            ],
                            aspect="auto",
                            interpolation="bilinear",
                            cmap=DENSITY_CMAP,
                            zorder=1
                        )

                        colorbar = fig.colorbar(
                            density_artist,
                            ax=ax,
                            fraction=0.046,
                            pad=0.035
                        )

                        format_colorbar(
                            colorbar,
                            "Kernel density"
                        )

                    except np.linalg.LinAlgError:

                        ax.scatter(
                            y_true,
                            y_pred,
                            s=12,
                            alpha=0.60,
                            edgecolors="none",
                            rasterized=True,
                            zorder=2
                        )

                else:
                    raise ValueError(
                        "mode must be either 'hexbin' or 'kde'."
                    )

                # --------------------------------------------
                # Running median and interquartile band
                # --------------------------------------------

                if show_running:

                    (
                        x_centers,
                        y_medians,
                        y_lower,
                        y_upper
                    ) = calculate_running_band(
                        y_true,
                        y_pred,
                        nbins=running_bins,
                        qlo=interquartile_range[0],
                        qhi=interquartile_range[1]
                    )

                    if x_centers.size > 0:

                        ax.fill_between(
                            x_centers,
                            y_lower,
                            y_upper,
                            color=RUNNING_LINE_COLOR,
                            alpha=0.18,
                            linewidth=0,
                            zorder=3
                        )

                        ax.plot(
                            x_centers,
                            y_medians,
                            color=RUNNING_LINE_COLOR,
                            linewidth=RUNNING_LINE_WIDTH,
                            linestyle="-",
                            label="Running median",
                            zorder=5
                        )

                # --------------------------------------------
                # Performance metrics
                # --------------------------------------------

                if show_metrics:

                    metrics = calculate_metrics(
                        y_true,
                        y_pred
                    )

                    metric_text = (
                        f"$R^2$ = "
                        f"{format_metric_value(metrics['R2'])}\n"
                        f"RMSE = "
                        f"{format_metric_value(metrics['RMSE'])}\n"
                        f"MAE = "
                        f"{format_metric_value(metrics['MAE'])}"
                    )

                    ax.text(
                        0.97,
                        0.04,
                        metric_text,
                        transform=ax.transAxes,
                        ha="right",
                        va="bottom",
                        fontsize=METRIC_FONT_SIZE,
                        color="black",
                        bbox={
                            "boxstyle": "square,pad=0.35",
                            "facecolor": "white",
                            "edgecolor": FRAME_COLOR,
                            "linewidth": 0.8,
                            "alpha": 0.90
                        },
                        zorder=8
                    )

                # --------------------------------------------
                # Titles and labels
                # --------------------------------------------

                target_title = TARGET_TITLES.get(
                    target,
                    target
                )

                unit = TARGET_UNITS.get(
                    target,
                    ""
                )

                ax.set_title(
                    target_title,
                    pad=10
                )

                if unit:
                    ax.set_xlabel(
                        f"Observed ({unit})",
                        labelpad=7
                    )

                    ax.set_ylabel(
                        f"Predicted ({unit})",
                        labelpad=7
                    )
                else:
                    ax.set_xlabel(
                        "Observed",
                        labelpad=7
                    )

                    ax.set_ylabel(
                        "Predicted",
                        labelpad=7
                    )

                # Identical x and y limits are essential
                # for a valid parity plot.
                ax.set_xlim(
                    lower_limit,
                    upper_limit
                )

                ax.set_ylim(
                    lower_limit,
                    upper_limit
                )

                ax.set_aspect(
                    "equal",
                    adjustable="box"
                )

                apply_manuscript_axis_style(ax)

                # --------------------------------------------
                # Panel label
                # --------------------------------------------

                if panel_number < len(panel_letters):

                    ax.text(
                        0.025,
                        0.975,
                        f"({panel_letters[panel_number]})",
                        transform=ax.transAxes,
                        fontsize=PANEL_LABEL_SIZE,
                        fontweight="bold",
                        ha="left",
                        va="top",
                        color="black",
                        bbox={
                            "facecolor": "white",
                            "edgecolor": "none",
                            "alpha": 0.80,
                            "pad": 1.5
                        },
                        zorder=10
                    )

                # --------------------------------------------
                # Optional row labels for multiple bandsets
                # --------------------------------------------

                if (
                    n_rows > 1
                    and column_index == 0
                ):

                    ax.text(
                        -0.30,
                        0.50,
                        bandset,
                        transform=ax.transAxes,
                        fontsize=ROW_LABEL_SIZE,
                        fontweight="semibold",
                        rotation=90,
                        ha="center",
                        va="center"
                    )

        # ----------------------------------------------------
        # Export
        # ----------------------------------------------------

        fig.savefig(
            OUTPUT_PNG,
            dpi=OUTPUT_DPI,
            bbox_inches="tight",
            facecolor=BACKGROUND_COLOR
        )

        fig.savefig(
            OUTPUT_PDF,
            bbox_inches="tight",
            facecolor=BACKGROUND_COLOR
        )

        plt.show()

        print(f"Saved PNG: {OUTPUT_PNG}")
        print(f"Saved PDF: {OUTPUT_PDF}")

        return fig, axes


# ============================================================
# 10. Run
# ============================================================

PREDS_STORE = preds_store_rf

# Optional diagnostic:
# print_available_prediction_keys(PREDS_STORE)

fig, axes = parity_density_grid(
    preds_store=PREDS_STORE,
    bandsets=BANDSETS,
    targets=TARGETS,
    mode="hexbin",
    gridsize=45,
    bins=None,
    show_running=True,
    running_bins=25,
    interquartile_range=(25, 75),
    show_metrics=True
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import matplotlib as mpl

# ============================================================
# CONFIG
# ============================================================


TARGETS = [
    "Volume",
    "n_trees",
    "max_h"
]

BANDSETS = [
    "AEF embeddings"
]

TARGET_LABELS = {
    "Volume": "Standing Volume (m³ ha⁻¹)",
    "n_trees": "Tree density (n ha⁻¹)",
    "max_h": "Canopy height (m)"
}

# ============================================================
# METRICS
# ============================================================

def _metrics(target, bandset):
    # This part assumes all_model_metrics_df exists, but it was not defined in the current context.
    # For now, it will return NaN. If this is a problem later, all_model_metrics_df needs to be created.
    # As per the problem statement, I'm focusing on fixing the NameError for preds_store_rf.
    return {
        "R2": np.nan,
        "RMSE": np.nan,
        "MAE": np.nan
    }

def _running_band(x, y, nbins=25, qlo=25, qhi=75):
    x = np.asarray(x)
    y = np.asarray(y)

    edges = np.quantile(x, np.linspace(0, 1, nbins + 1))
    edges[0] -= 1e-9
    edges[-1] += 1e-9

    idx = np.digitize(x, edges) - 1

    xc, ym, ylo, yhi = [], [], [], []

    for b in range(nbins):
        m = idx == b
        if m.any():
            xc.append(np.median(x[m]))
            ym.append(np.median(y[m]))
            ylo.append(np.percentile(y[m], qlo))
            yhi.append(np.percentile(y[m], qhi))

    return (
        np.array(xc),
        np.array(ym),
        np.array(ylo),
        np.array(yhi)
    )


def print_available_pred_bandsets(preds_store, targets=TARGETS):
    print("\n=== Available bandset keys in PREDS_STORE ===")

    for target in targets:
        keys = sorted(list(preds_store.get(target, {}).keys()))
        print(f"{target}:")

        if keys:
            for key in keys:
                print(f"  - {key}")
        else:
            print("  (none)")


def _get_store(preds_store, target, bandset):
    return preds_store.get(target, {}).get(bandset, None)

# Removed the call to print_available_pred_bandsets here, it will be called in a separate cell after PREDS_STORE is defined

# ============================================================
# PLOT FUNCTION
# ============================================================

def parity_density_grid(
    preds_store,
    bandsets=BANDSETS,
    targets=TARGETS,
    mode="hexbin",
    gridsize=45,
    bins=None, # Changed default from "log" to None
    kde_res=200,
    show_running=True,
    nbins_run=25,
    iqr=(25, 75),
    figsize_per=(4.8, 4.8)
):

    with mpl.rc_context({
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
        "font.size": 14,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12
    }):

        nrows = len(bandsets)
        ncols = len(targets)

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(figsize_per[0] * ncols, figsize_per[1] * nrows),
            squeeze=False
        )

        for i, bandset in enumerate(bandsets):

            for j, target in enumerate(targets):

                ax = axes[i, j]

                store = _get_store(
                    preds_store,
                    target,
                    bandset
                )

                if store is None or store.get("y_true", np.array([])).size == 0:
                    ax.axis("off")
                    ax.text(
                        0.5,
                        0.5,
                        "no preds",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                        fontsize=16
                    )
                    continue

                y_true = np.asarray(store["y_true"]).ravel()
                y_pred = np.asarray(store["y_pred"]).ravel()

                valid = np.isfinite(y_true) & np.isfinite(y_pred)
                y_true = y_true[valid]
                y_pred = y_pred[valid]

                lo = float(min(y_true.min(), y_pred.min()))
                hi = float(max(y_true.max(), y_pred.max()))

                pad = 0.05 * (hi - lo)
                lo -= pad
                hi += pad

                ax.plot(
                    [lo, hi],
                    [lo, hi],
                    color="black",
                    lw=1.2,
                    linestyle="-"
                )

                if mode == "hexbin":
                    hb = ax.hexbin(
                      y_true,
                      y_pred,
                      gridsize=gridsize,
                      bins=bins, # Use the bins parameter from function argument (now defaults to None)
                      mincnt=1,
                      cmap="RdYlBu_r"
                  )
                    # Only add colorbar for the rightmost graph in each row
                    if j == ncols - 1: # Condition for the rightmost column
                        fig.colorbar(
                            hb,
                            ax=ax,
                            fraction=0.046,
                            pad=0.02,
                            location='right' # Changed from 'left' to 'right'
                        ).set_label("Density", fontsize=12)

                elif mode == "kde":
                    xx, yy = np.mgrid[
                        lo:hi:complex(kde_res),
                        lo:hi:complex(kde_res)
                    ]

                    kde = gaussian_kde(
                        np.vstack([y_true, y_pred])
                    )

                    zz = kde(
                        np.vstack([xx.ravel(), yy.ravel()])
                    ).reshape(xx.shape)

                    im = ax.imshow(
                        zz.T,
                        origin="lower",
                        extent=[lo, hi, lo, hi],
                        aspect="equal",
                        interpolation="bilinear"
                    )

                    fig.colorbar(
                        im,
                        ax=ax,
                        fraction=0.046,
                        pad=0.02
                    ).set_label("Density", fontsize=12)

                if show_running:
                    xc, ym, ylo, yhi = _running_band(
                        y_true,
                        y_pred,
                        nbins=nbins_run,
                        qlo=iqr[0],
                        qhi=iqr[1]
                    )

                    ax.plot(
                        xc,
                        ym,
                        color="red",
                        lw=1.8
                    )

                    ax.fill_between(
                        xc,
                        ylo,
                        yhi,
                        color="red",
                        alpha=0.18,
                        linewidth=0
                    )

                m = _metrics(target, bandset)



                ax.set_title(
                    TARGET_LABELS.get(target, target),
                    fontweight="bold"
                )

                ax.set_xlabel("Observed")
                ax.set_ylabel("Predicted")

                ax.set_xlim(lo, hi)
                ax.set_ylim(lo, hi)
                ax.set_aspect("equal", "box")

        plt.tight_layout()

        plt.savefig(
            "parity_density_AEF_volume_n_tree_max_h.png",
            dpi=600,
            bbox_inches="tight"
        )

        plt.show()

# ============================================================
# RUN
# ============================================================

PREDS_STORE = preds_store_rf
parity_density_grid(
    PREDS_STORE,
    mode="hexbin",
    gridsize=45,
    bins=None, # Changed from "log" to None to improve number formatting
    show_running=True,
    nbins_run=25
)


## Error analysis

### Residual Analysis: Absolute Residuals binned by Actual Target Values

Below are the residual plots, where the absolute residuals for each target ('Volume', 'n_trees', 'max_h') are binned according to the actual observed values of that specific target. This visualization helps in understanding if the model's error varies across the range of the observed values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# ========================================================
# CONFIGURATION
# ========================================================

plt.rcParams.update({
    "figure.autolayout": True,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
    "font.family": "sans-serif",
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
})

TARGET_LABELS = {
    "Volume": "Volume (m³ ha⁻¹)",
    "n_trees": "Tree density (n ha⁻¹)",
    "max_h": "Canopy height (m)"
}

# ========================================================
# HELPER FUNCTIONS
# ========================================================

def _get_bins(data, n_bins=10):
    return pd.qcut(
        data,
        q=n_bins,
        labels=False,
        duplicates="drop",
        retbins=True
    )


def _get_metrics(target):
    metrics_row = results_df[
        (results_df["Target"] == target) &
        (results_df["Pass"].str.contains("Outliers removed", na=False))
    ]

    if not metrics_row.empty:
        r2 = metrics_row["R2_mean"].iloc[0]
        rmse = metrics_row["RMSE_mean"].iloc[0]
        mae = metrics_row["MAE_mean"].iloc[0]
        return f"R²={r2:.2f}, RMSE={rmse:.2f}, MAE={mae:.2f}"

    return "Metrics N/A"


def plot_boxplot_grid_by_structure_bins(
    residual_df,
    bin_feature_prefix,
    bandsets,
    targets,
    n_bins=10,
    figsize_per_plot=(5.5, 4.5),
    save_path=None
):
    """
    Plot absolute residuals binned by observed structural variables.
    Layout: 1 row × 3 columns.
    """

    n_cols = 3
    n_rows = 1

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_plot[0] * n_cols, figsize_per_plot[1]),
        squeeze=False
    )

    axes = axes.flatten()

    plot_idx = 0

    for target in targets:
        for bandset in bandsets:

            if plot_idx >= len(axes):
                break

            ax = axes[plot_idx]
            target_label = TARGET_LABELS.get(target, target)

            current_bin_feature = (
                f"{bin_feature_prefix}{target.lower().replace(' ', '_')}"
            )

            plot_data = residual_df[
                (residual_df["target"] == target) &
                (residual_df["bandset"] == bandset)
            ].copy()

            if current_bin_feature not in plot_data.columns:
                ax.text(
                    0.5, 0.5,
                    f"Bin feature missing:\n{current_bin_feature}",
                    ha="center",
                    va="center",
                    transform=ax.transAxes
                )
                ax.set_title(f"{target_label}\nBin Feature Missing")
                ax.set_xticks([])
                ax.set_yticks([])
                plot_idx += 1
                continue

            plot_data = plot_data.dropna(
                subset=[current_bin_feature, "abs_residual"]
            )

            if plot_data.empty:
                ax.text(
                    0.5, 0.5,
                    f"No data for {target_label}",
                    ha="center",
                    va="center",
                    transform=ax.transAxes
                )
                ax.set_title(f"{target_label}\nNo Data")
                ax.set_xticks([])
                ax.set_yticks([])
                plot_idx += 1
                continue

            plot_data["bin"], bin_edges = _get_bins(
                plot_data[current_bin_feature],
                n_bins=n_bins
            )

            bin_labels = [
                f"{bin_edges[i]:.0f}–{bin_edges[i + 1]:.0f}"
                for i in range(len(bin_edges) - 1)
            ]

            unique_bins = sorted(plot_data["bin"].dropna().unique())

            bin_map = {
                b: bin_labels[int(b)]
                for b in unique_bins
                if int(b) < len(bin_labels)
            }

            plot_data["bin_label"] = plot_data["bin"].map(bin_map)

            plot_data["bin_label"] = pd.Categorical(
                plot_data["bin_label"],
                categories=bin_labels,
                ordered=True
            )

            sns.boxplot(
                x="bin_label",
                y="abs_residual",
                data=plot_data,
                ax=ax,
                color="lightblue",
                fliersize=3,
                linewidth=0.8
            )



            ax.set_xlabel(
                f"Observed {target_label}",
                fontsize=16
            )

            # Only show y-axis label for the leftmost plot
            if plot_idx == 0:
                ax.set_ylabel(
                    "Absolute residual",
                    fontsize=16
                )
            else:
                ax.set_ylabel('') # Set empty label for other plots

            ax.tick_params(axis="x", rotation=45)
            ax.grid(True, linestyle="--", alpha=0.6)

            plot_idx += 1

    for i in range(plot_idx, len(axes)):
        fig.delaxes(axes[i])

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()


# ========================================================
# RUN PLOT
# ========================================================

TARGETS_FOR_RESIDUAL_PLOTS = ["Volume", "n_trees", "max_h"]

plot_boxplot_grid_by_structure_bins(
    residual_df=residual_df_for_plotting,
    bin_feature_prefix="actual_",
    bandsets=["AEF embeddings"],
    targets=TARGETS_FOR_RESIDUAL_PLOTS,
    n_bins=10,
    figsize_per_plot=(5.8, 4.8),
    save_path="residual_boxplots_AEF_1row_3columns.png"
)


In [ ]:
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# 1. Configuration
# ============================================================

TARGETS = [
    "Volume",
    "n_trees",
    "max_h"
]

BANDSETS = [
    "AEF embeddings"
]

TARGET_TITLES = {
    "Volume": "Standing volume",
    "n_trees": "Tree density",
    "max_h": "Canopy height"
}

TARGET_UNITS = {
    "Volume": "m³ ha⁻¹",
    "n_trees": "trees ha⁻¹",
    "max_h": "m"
}

# Number of decimal places used in bin labels
BIN_DECIMALS = {
    "Volume": 0,
    "n_trees": 0,
    "max_h": 1
}

# ------------------------------------------------------------
# Manuscript style
# ------------------------------------------------------------

FONT_FAMILY = "sans-serif"

BASE_FONT_SIZE = 12
TITLE_FONT_SIZE = 15
AXIS_LABEL_SIZE = 13
TICK_LABEL_SIZE = 10
PANEL_LABEL_SIZE = 13
ROW_LABEL_SIZE = 13

FRAME_LINE_WIDTH = 1.25
TICK_LINE_WIDTH = 1.10
GRID_LINE_WIDTH = 0.70

BOX_LINE_WIDTH = 1.10
MEDIAN_LINE_WIDTH = 1.60

BACKGROUND_COLOR = "white"
FRAME_COLOR = "black"
GRID_COLOR = "#D9D9D9"

# Colorblind-friendly box color
BOX_COLOR = "#56B4E9"
MEDIAN_COLOR = "#D55E00"
OUTLIER_COLOR = "#595959"

FIGURE_WIDTH_PER_PANEL = 5.6
FIGURE_HEIGHT_PER_PANEL = 4.9

OUTPUT_PNG = "residual_boxplots_AEF_harmonized.png"
OUTPUT_PDF = "residual_boxplots_AEF_harmonized.pdf"
OUTPUT_DPI = 600


plt.rcParams.update({
    # Fonts
    "font.family": FONT_FAMILY,
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": BASE_FONT_SIZE,

    "axes.titlesize": TITLE_FONT_SIZE,
    "axes.titleweight": "semibold",

    "axes.labelsize": AXIS_LABEL_SIZE,
    "axes.labelweight": "semibold",

    "xtick.labelsize": TICK_LABEL_SIZE,
    "ytick.labelsize": TICK_LABEL_SIZE,

    # Backgrounds
    "figure.facecolor": BACKGROUND_COLOR,
    "axes.facecolor": BACKGROUND_COLOR,
    "savefig.facecolor": BACKGROUND_COLOR,

    # Complete frames
    "axes.edgecolor": FRAME_COLOR,
    "axes.linewidth": FRAME_LINE_WIDTH,
    "axes.spines.top": True,
    "axes.spines.right": True,

    # Gridlines
    "axes.grid": False, # Modified: Set to False to remove grid
    "axes.axisbelow": True,
    "grid.color": GRID_COLOR,
    "grid.linestyle": "-",
    "grid.linewidth": GRID_LINE_WIDTH,
    "grid.alpha": 0.75,

    # Tick marks
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": TICK_LINE_WIDTH,
    "ytick.major.width": TICK_LINE_WIDTH,
    "xtick.major.size": 5,
    "ytick.major.size": 5,

    # Editable fonts in vector outputs
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


# ============================================================
# 2. Reusable axis-formatting function
# ============================================================

def apply_manuscript_axis_style(ax):
    """
    Apply the same background, grid, complete frame, and
    tick formatting used in the other manuscript figures.
    """

    ax.set_facecolor(BACKGROUND_COLOR)
    ax.set_axisbelow(True)

    # Major gridlines only
    # ax.grid(
    #     visible=True,
    #     which="major",
    #     axis="both",
    #     color=GRID_COLOR,
    #     linestyle="-",
    #     linewidth=GRID_LINE_WIDTH,
    #     alpha=0.75
    # )
    # Modified: Removed ax.grid call to ensure no grid is shown

    ax.grid(False, which="minor")
    ax.minorticks_off()

    # Complete rectangular frame
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(FRAME_COLOR)
        spine.set_linewidth(FRAME_LINE_WIDTH)

    # Standardized ticks
    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=5,
        width=TICK_LINE_WIDTH,
        color=FRAME_COLOR,
        labelcolor="black",
        top=False,
        right=False
    )


# ============================================================
# 3. Quantile-bin helper
# ============================================================

def create_quantile_bins(
    data,
    value_column,
    target,
    n_bins=10
):
    """
    Divide observations into approximately equal-sized
    quantile bins and create readable range labels.
    """

    working_data = data.copy()

    bin_numbers, bin_edges = pd.qcut(
        working_data[value_column],
        q=n_bins,
        labels=False,
        duplicates="drop",
        retbins=True
    )

    working_data["bin_number"] = bin_numbers

    number_of_actual_bins = len(bin_edges) - 1

    decimals = BIN_DECIMALS.get(target, 1)

    bin_labels = []

    for index in range(number_of_actual_bins):

        lower = bin_edges[index]
        upper = bin_edges[index + 1]

        label = (
            f"{lower:.{decimals}f}–"
            f"{upper:.{decimals}f}"
        )

        bin_labels.append(label)

    bin_mapping = {
        bin_number: bin_labels[int(bin_number)]
        for bin_number in sorted(
            working_data["bin_number"]
            .dropna()
            .unique()
        )
        if int(bin_number) < len(bin_labels)
    }

    working_data["bin_label"] = (
        working_data["bin_number"]
        .map(bin_mapping)
    )

    working_data["bin_label"] = pd.Categorical(
        working_data["bin_label"],
        categories=bin_labels,
        ordered=True
    )

    return working_data, bin_labels


# ============================================================
# 4. Empty-panel helper
# ============================================================

def format_empty_panel(
    ax,
    message,
    title,
    panel_label
):
    """
    Preserve the complete figure style even when a requested
    variable or prediction set is unavailable.
    """

    apply_manuscript_axis_style(ax)

    ax.text(
        0.5,
        0.5,
        message,
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=BASE_FONT_SIZE,
        color="#666666"
    )

    ax.set_title(
        title,
        pad=10
    )

    ax.set_xticks([])
    ax.set_yticks([])

    # ax.text(
    #     0.025,
    #     0.975,
    #     panel_label,
    #     transform=ax.transAxes,
    #     fontsize=PANEL_LABEL_SIZE,
    #     fontweight="bold",
    #     ha="left",
    #     va="top",
    #     color="black"
    # )


# ============================================================
# 5. Main plotting function
# ============================================================

def plot_boxplot_grid_by_structure_bins(
    residual_df,
    bin_feature_prefix,
    bandsets,
    targets,
    n_bins=10,
    save_png=OUTPUT_PNG,
    save_pdf=OUTPUT_PDF
):
    """
    Plot absolute residual distributions across quantile bins
    of observed forest structural attributes.
    """

    n_rows = len(bandsets)
    n_cols = len(targets)

    fig, axes = plt.subplots(
        nrows=n_rows,
        ncols=n_cols,
        figsize=(
            FIGURE_WIDTH_PER_PANEL * n_cols,
            FIGURE_HEIGHT_PER_PANEL * n_rows
        ),
        squeeze=False,
        facecolor=BACKGROUND_COLOR,
        constrained_layout=True
    )

    panel_letters = list(string.ascii_lowercase)

    for row_index, bandset in enumerate(bandsets):

        for column_index, target in enumerate(targets):

            ax = axes[row_index, column_index]

            panel_number = (
                row_index * n_cols
                + column_index
            )

            panel_label = (
                f"({panel_letters[panel_number]})"
            )

            target_title = TARGET_TITLES.get(
                target,
                target
            )

            target_unit = TARGET_UNITS.get(
                target,
                ""
            )

            current_bin_feature = (
                f"{bin_feature_prefix}"
                f"{target.lower().replace(' ', '_')}"
            )

            # ------------------------------------------------
            # Select the requested target and predictor set
            # ------------------------------------------------

            plot_data = residual_df[
                (residual_df["target"] == target)
                &
                (residual_df["bandset"] == bandset)
            ].copy()

            # ------------------------------------------------
            # Check required columns
            # ------------------------------------------------

            if current_bin_feature not in plot_data.columns:

                format_empty_panel(
                    ax=ax,
                    message=(
                        "Observed-value column unavailable:\n"
                        f"{current_bin_feature}"
                    ),
                    title=target_title,
                    panel_label=panel_label
                )

                continue

            if "abs_residual" not in plot_data.columns:

                format_empty_panel(
                    ax=ax,
                    message=(
                        "Column unavailable:\n"
                        "abs_residual"
                    ),
                    title=target_title,
                    panel_label=panel_label
                )

                continue

            # Convert to numeric values
            plot_data[current_bin_feature] = pd.to_numeric(
                plot_data[current_bin_feature],
                errors="coerce"
            )

            plot_data["abs_residual"] = pd.to_numeric(
                plot_data["abs_residual"],
                errors="coerce"
            )

            plot_data = plot_data.replace(
                [np.inf, -np.inf],
                np.nan
            )

            plot_data = plot_data.dropna(
                subset=[
                    current_bin_feature,
                    "abs_residual"
                ]
            )

            # Absolute residuals cannot be negative
            plot_data = plot_data[
                plot_data["abs_residual"] >= 0
            ].copy()

            if plot_data.empty:

                format_empty_panel(
                    ax=ax,
                    message="No valid residual observations",
                    title=target_title,
                    panel_label=panel_label
                )

                continue

            # ------------------------------------------------
            # Create target-specific quantile bins
            # ------------------------------------------------

            try:

                plot_data, bin_labels = (
                    create_quantile_bins(
                        data=plot_data,
                        value_column=current_bin_feature,
                        target=target,
                        n_bins=n_bins
                    )
                )

            except ValueError:

                format_empty_panel(
                    ax=ax,
                    message=(
                        "Insufficient unique observations\n"
                        "for quantile binning"
                    ),
                    title=target_title,
                    panel_label=panel_label
                )

                continue

            # Remove observations without assigned bins
            plot_data = plot_data.dropna(
                subset=["bin_label"]
            )

            if plot_data.empty:

                format_empty_panel(
                    ax=ax,
                    message="No observations after binning",
                    title=target_title,
                    panel_label=panel_label
                )

                continue

            # ------------------------------------------------
            # Draw the boxplots
            # ------------------------------------------------

            sns.boxplot(
                data=plot_data,
                x="bin_label",
                y="abs_residual",
                order=bin_labels,
                ax=ax,

                width=0.68,
                color=BOX_COLOR,
                saturation=1.0,

                showfliers=True,
                fliersize=2.8,

                linewidth=BOX_LINE_WIDTH,

                boxprops={
                    "facecolor": BOX_COLOR,
                    "edgecolor": FRAME_COLOR,
                    "linewidth": BOX_LINE_WIDTH,
                    "alpha": 0.82
                },

                whiskerprops={
                    "color": FRAME_COLOR,
                    "linewidth": BOX_LINE_WIDTH
                },

                capprops={
                    "color": FRAME_COLOR,
                    "linewidth": BOX_LINE_WIDTH
                },

                medianprops={
                    "color": MEDIAN_COLOR,
                    "linewidth": MEDIAN_LINE_WIDTH
                },

                flierprops={
                    "marker": "o",
                    "markersize": 2.8,
                    "markerfacecolor": OUTLIER_COLOR,
                    "markeredgecolor": OUTLIER_COLOR,
                    "markeredgewidth": 0,
                    "alpha": 0.45
                }
            )

            # ------------------------------------------------
            # Titles and axis labels
            # ------------------------------------------------

            ax.set_title(
                target_title,
                pad=10
            )

            if target_unit:

                ax.set_xlabel(
                    f"Observed {target_title.lower()} "
                    f"({target_unit})",
                    labelpad=8
                )

                ax.set_ylabel(
                    f"Absolute residual ({target_unit})",
                    labelpad=8
                )

            else:

                ax.set_xlabel(
                    f"Observed {target_title.lower()}",
                    labelpad=8
                )

                ax.set_ylabel(
                    "Absolute residual",
                    labelpad=8
                )

            # Absolute residuals begin at zero
            ax.set_ylim(bottom=0)

            # Rotate crowded quantile-range labels
            ax.tick_params(
                axis="x",
                labelrotation=45
            )

            for tick_label in ax.get_xticklabels():
                tick_label.set_horizontalalignment("right")
                tick_label.set_rotation_mode("anchor")

            apply_manuscript_axis_style(ax)

            # ------------------------------------------------
            # Panel label
            # ------------------------------------------------

            # ax.text(
            #     0.025,
            #     0.975,
            #     panel_label,
            #     transform=ax.transAxes,
            #     fontsize=PANEL_LABEL_SIZE,
            #     fontweight="bold",
            #     ha="left",
            #     va="top",
            #     color="black",
            #     bbox={
            #         "facecolor": "white",
            #         "edgecolor": "none",
            #         "alpha": 0.80,
            #         "pad": 1.5
            #     },
            #     zorder=10
            # )

            # ------------------------------------------------
            # Optional row labels when comparing bandsets
            # ------------------------------------------------

            if (
                n_rows > 1
                and column_index == 0
            ):

                ax.text(
                    -0.30,
                    0.50,
                    bandset,
                    transform=ax.transAxes,
                    fontsize=ROW_LABEL_SIZE,
                    fontweight="semibold",
                    rotation=90,
                    ha="center",
                    va="center"
                )

    # ========================================================
    # 6. Export
    # ========================================================

    if save_png is not None:

        fig.savefig(
            save_png,
            dpi=OUTPUT_DPI,
            bbox_inches="tight",
            facecolor=BACKGROUND_COLOR
        )

        print(f"Saved PNG: {save_png}")

    if save_pdf is not None:

        fig.savefig(
            save_pdf,
            bbox_inches="tight",
            facecolor=BACKGROUND_COLOR
        )

        print(f"Saved PDF: {save_pdf}")

    plt.show()

    return fig, axes


# ============================================================
# 7. Run
# ============================================================

TARGETS_FOR_RESIDUAL_PLOTS = [
    "Volume",
    "n_trees",
    "max_h"
]

fig, axes = plot_boxplot_grid_by_structure_bins(
    residual_df=residual_df_for_plotting,
    bin_feature_prefix="actual_",
    bandsets=["AEF embeddings"],
    targets=TARGETS_FOR_RESIDUAL_PLOTS,
    n_bins=10,
    save_png=OUTPUT_PNG,
    save_pdf=OUTPUT_PDF
)

# Predicting the unmeasured

### Descriptive Statistics for Predicted Values

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# ==========================================================
# Accessing variables from previous cells (HBNeD6ZEizwn)
# ==========================================================
# df_base is merged_embeddings_field_dem_bioclim
# TARGETS_WANTED = ["Volume", "n_trees", "max_h"]
# AEF_COLS = list of AEF embedding columns
# OUTLIER_SD_THRESH = 0.8

# Ensure df_base is the latest merged_embeddings_field_dem_bioclim
if 'merged_embeddings_field_dem_bioclim' in locals():
    df_base = merged_embeddings_field_dem_bioclim.copy()
else:
    print("Error: merged_embeddings_field_dem_bioclim not found. Please run previous cells.")
    # Fallback or error handling for if the dataframe isn't available
    # For this example, we'll assume it exists or raise an error.
    raise ValueError("Required dataframe 'merged_embeddings_field_dem_bioclim' is not available.")

# Ensure AEF_COLS is defined
if 'AEF_COLS' not in locals():
    # Re-define AEF_COLS if not already in scope (e.g., if this cell is run independently)
    AEF_COLS = [
        c for c in df_base.columns
        if len(c) == 3 and c[0] == "A" and c[1:].isdigit()
    ]
    AEF_COLS = sorted(
        AEF_COLS,
        key=lambda x: int(x[1:]) if isinstance(x, str) and x[1:].isdigit() else 10**9
    )

# Define run_cv function if not already defined (or ensure it's imported/available)
def _sort_A(cols):
    return sorted(
        cols,
        key=lambda x: int(x[1:]) if isinstance(x, str) and x[1:].isdigit() else 10**9
    )

# Re-define run_cv for this cell's scope if needed (avoiding re-execution side effects)
# This is a simplified version just to get residuals for the *measured* data's outlier removal
from sklearn.metrics import r2_score, mean_absolute_error
def run_cv_for_outlier_detection(df, target, features):
    df_filtered = df[df[target].notna()].copy()
    group_col = "PLOTID" if "PLOTID" in df_filtered.columns else "code"

    X = df_filtered[features]
    y = df_filtered[target]
    groups = df_filtered[group_col]

    gkf = GroupKFold(n_splits=5)
    residuals = pd.Series(np.nan, index=df_filtered.index)

    for fold_num, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        if len(train_idx) == 0 or len(test_idx) == 0:
            print(f"      Warning: Skipping fold {fold_num} for target {target} due to empty train or test split. (Train samples: {len(train_idx)}, Test samples: {len(test_idx)})")
            continue

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        imp = SimpleImputer(strategy="median")
        X_train_imputed = imp.fit_transform(X_train)
        X_test_imputed  = imp.transform(X_test)

        model = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
        model.fit(X_train_imputed, y_train)
        y_pred = model.predict(X_test_imputed)

        test_index = df_filtered.index[test_idx]
        residuals.loc[test_index] = np.abs(y_test.values - y_pred) # Absolute residuals

    return residuals, df_filtered


# Create a copy of the base dataframe to store predictions
final_predictions_df = df_base.copy()

# Identify unmeasured rows for prediction
df_unmeasured_subset = df_base[df_base['plot_status'] == 'unmeasured'].copy()
# Features for unmeasured data, will be imputed later
X_unmeasured_features = df_unmeasured_subset[AEF_COLS]

print("Estimating target values for unmeasured plots...")

for target in TARGETS_WANTED:
    print(f"  Processing target: {target}")

    # 1. Identify measured data for this target and remove NaNs
    df_measured_for_target = df_base[
        (df_base['plot_status'] == 'measured') &
        (df_base[target].notna())
    ].copy()

    if df_measured_for_target.empty:
        print(f"    No measured data available for {target}. Skipping.")
        continue

    # 2. Get residuals from the measured data (Pass 1 of the CV process)
    # This allows us to apply the same outlier removal logic as in the training.
    residuals_pass1, df_filtered_pass1 = run_cv_for_outlier_detection(df_measured_for_target, target, AEF_COLS)

    # 3. Apply outlier removal logic to get the 'cleaned' training data
    res_mean = residuals_pass1.mean()
    res_sd = residuals_pass1.std()
    threshold = res_mean + OUTLIER_SD_THRESH * res_sd

    # Filter the original indices based on the residuals series' index
    # Only keep indices where a residual was actually calculated (i.e., not NaN)
    valid_residual_indices = residuals_pass1.dropna().index
    if not valid_residual_indices.empty:
        df_final_training = df_filtered_pass1.loc[valid_residual_indices][residuals_pass1.loc[valid_residual_indices] <= threshold]
    else:
        df_final_training = pd.DataFrame()

    print(f"    Original measured N for {target}: {len(df_measured_for_target)}")
    print(f"    Cleaned measured N for {target} (after outlier removal): {len(df_final_training)}")

    if df_final_training.empty:
        print(f"    Warning: Cleaned training data for {target} is empty after outlier removal. Skipping prediction for this target.")
        continue

    # 4. Train the final RandomForestRegressor on the cleaned measured data
    X_train = df_final_training[AEF_COLS]
    y_train = df_final_training[target]

    # Fit imputer and model on the final training set
    imputer = SimpleImputer(strategy="median")
    X_train_imputed = imputer.fit_transform(X_train)

    model = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    model.fit(X_train_imputed, y_train)

    # 5. Prepare unmeasured data for prediction using the same imputer
    # Need to handle potential NaNs in unmeasured features. Use the same imputer fitted on training data.
    X_unmeasured_imputed = imputer.transform(X_unmeasured_features)

    # 6. Predict and assign to the final_predictions_df
    predictions = model.predict(X_unmeasured_imputed)

    # Assign predictions back to the original index of the unmeasured rows
    final_predictions_df.loc[df_unmeasured_subset.index, f'predicted_{target.lower()}'] = predictions

print("\nPrediction complete. Displaying a sample of unmeasured rows with new predictions.")

# Display only the unmeasured rows with original target and new prediction columns
predicted_cols = [f'predicted_{t.lower()}' for t in TARGETS_WANTED]
columns_to_display = ['PLOTID', 'plot_status'] + TARGETS_WANTED + predicted_cols

display(final_predictions_df[
    final_predictions_df['plot_status'] == 'unmeasured'
][columns_to_display].head())

# Update the main df_base variable with new predictions
merged_embeddings_field_dem_bioclim = final_predictions_df.copy()

print(f"Shape of updated merged_embeddings_field_dem_bioclim: {merged_embeddings_field_dem_bioclim.shape}")

In [ ]:
output_csv_path_with_predictions = str(PREDICTION_OUTPUT_PATH)

merged_embeddings_field_dem_bioclim.to_csv(output_csv_path_with_predictions, index=False)

print(f"Updated DataFrame saved successfully to: {output_csv_path_with_predictions}")

In [ ]:
predicted_cols = [
    'predicted_volume',
    'predicted_n_trees',
    'predicted_max_h'
]

# Ensure these columns exist in the DataFrame
existing_predicted_cols = [col for col in predicted_cols if col in merged_embeddings_field_dem_bioclim.columns]

if existing_predicted_cols:
    print("Descriptive statistics for predicted values:")
    display(merged_embeddings_field_dem_bioclim[existing_predicted_cols].describe())
else:
    print("No predicted columns found in the DataFrame.")

### Descriptive Statistics for Predicted Values by Plot Status (Measured vs. Unmeasured)

In [ ]:
predicted_cols = [
    'predicted_volume',
    'predicted_n_trees',
    'predicted_max_h'
]

# 1. Descriptive statistics for 'measured' plots using predicted columns
print("Descriptive statistics for predicted values in 'measured' plots:")
measured_plots_predicted_stats = merged_embeddings_field_dem_bioclim[
    merged_embeddings_field_dem_bioclim['plot_status'] == 'measured'
][predicted_cols].describe()
display(measured_plots_predicted_stats)

# 2. Descriptive statistics for 'unmeasured' plots using predicted columns
print("\nDescriptive statistics for predicted values in 'unmeasured' plots:")
unmeasured_plots_predicted_stats = merged_embeddings_field_dem_bioclim[
    merged_embeddings_field_dem_bioclim['plot_status'] == 'unmeasured'
][predicted_cols].describe()
display(unmeasured_plots_predicted_stats)

# 3. Descriptive statistics for all plots combined using predicted columns
print("\nDescriptive statistics for predicted values (all plots combined):")
# This will effectively calculate statistics only for non-NaN values, which are from unmeasured plots.
all_plots_predicted_stats = merged_embeddings_field_dem_bioclim[predicted_cols].describe()
display(all_plots_predicted_stats)

In [ ]:
for target in TARGETS_WANTED:
    new_col_name = f'final_{target.lower()}'
    predicted_col_name = f'predicted_{target.lower()}'

    # Initialize the new column with NaN
    merged_embeddings_field_dem_bioclim[new_col_name] = np.nan

    # Fill with measured values for 'measured' plots
    measured_mask = merged_embeddings_field_dem_bioclim['plot_status'] == 'measured'
    merged_embeddings_field_dem_bioclim.loc[measured_mask, new_col_name] = \
        merged_embeddings_field_dem_bioclim.loc[measured_mask, target]

    # Fill with predicted values for 'unmeasured' plots
    unmeasured_mask = merged_embeddings_field_dem_bioclim['plot_status'] == 'unmeasured'
    merged_embeddings_field_dem_bioclim.loc[unmeasured_mask, new_col_name] = \
        merged_embeddings_field_dem_bioclim.loc[unmeasured_mask, predicted_col_name]

print("New 'final_' columns created combining measured and predicted values.")

# Display relevant columns to show the result
display_cols = ['PLOTID', 'plot_status'] + TARGETS_WANTED + [f'predicted_{t.lower()}' for t in TARGETS_WANTED] + [f'final_{t.lower()}' for t in TARGETS_WANTED]
display(merged_embeddings_field_dem_bioclim[display_cols].head())

In [ ]:
import numpy as np
from scipy import stats

final_cols = ['final_volume', 'final_n_trees', 'final_max_h']

# Ensure these columns exist in the DataFrame
existing_final_cols = [col for col in final_cols if col in merged_embeddings_field_dem_bioclim.columns]

if existing_final_cols:
    stats_data = []
    for col in existing_final_cols:
        data = merged_embeddings_field_dem_bioclim[col].dropna()
        if not data.empty:
            mean_val = data.mean()
            std_val = data.std()
            count_val = len(data)

            # Calculate 95% Confidence Interval
            if count_val > 1:
                sem = std_val / np.sqrt(count_val)  # Standard Error of the Mean
                # Use a t-distribution for CI calculation (more accurate for smaller n)
                confidence_interval = stats.t.interval(0.95, df=count_val - 1, loc=mean_val, scale=sem)
                ci_str = f"({confidence_interval[0]:.2f} - {confidence_interval[1]:.2f})"
            else:
                ci_str = "(N/A)"

            stats_data.append({
                "Structural attribute": col.replace('final_', '').replace('_', ' ').title(),
                "Mean": f"{mean_val:.2f}",
                "SD": f"{std_val:.2f}",
                "n": int(count_val),
                "CI": ci_str
            })

    if stats_data:
        summary_df = pd.DataFrame(stats_data)
        display(summary_df)
    else:
        print("No data available to compute statistics for the 'final_' columns.")
else:
    print("No 'final_' columns found in the DataFrame.")

# Mapping

## Designing 10*10 m Grid

In [ ]:
import ee
import os

# ==========================================================
# Initialize Earth Engine
# ==========================================================

try:
    ee.Initialize(project='YOUR_GCP_PROJECT_ID')
except:
    ee.Authenticate(project='YOUR_GCP_PROJECT_ID')
    ee.Initialize(project='YOUR_GCP_PROJECT_ID')

# ==========================================================
# Parameters
# ==========================================================

GRID_RESOLUTION_METERS = 10
BATCH_SIZE = 10000

# ==========================================================
# Create Grid
# ==========================================================

print("Creating grid...")

grid_image = ee.Image.pixelLonLat().addBands(
    ee.Image.constant(1).rename('constant')
)

grid_points_fc = (
    grid_image.sample(
        region=aoi_geom,
        scale=GRID_RESOLUTION_METERS,
        projection='EPSG:4326',
        geometries=True,
        tileScale=4
    )
    .map(lambda f: f.set({
        'longitude': f.geometry().coordinates().get(0),
        'latitude': f.geometry().coordinates().get(1)
    }))
)

grid_points_fc = grid_points_fc.map(
    lambda f: f.set('grid_id', f.id())
)

num_grid_points = grid_points_fc.size().getInfo()

print(f"Generated {num_grid_points:,} grid points")

# ==========================================================
# Export Embeddings
# ==========================================================

points_list = grid_points_fc.toList(num_grid_points)

for i in range(0, num_grid_points, BATCH_SIZE):

    end_index = min(i + BATCH_SIZE, num_grid_points)

    batch_points = ee.FeatureCollection(
        points_list.slice(i, end_index)
    )

    sampled_batch = embedding2023.sampleRegions(
        collection=batch_points,
        properties=[
            'grid_id',
            'longitude',
            'latitude'
        ],
        scale=10,
        geometries=False,
        tileScale=4
    )

    description = (
        f'GridEmbeddings_Batch_{i}_to_{end_index-1}'
    )

    task = ee.batch.Export.table.toDrive(
        collection=sampled_batch,
        description=description,
        folder='AlphaEarth_MBI_exports',
        fileNamePrefix=description,
        fileFormat='CSV'
    )

    task.start()

    print(f"🚀 Started: {description}")

print("\nAll export tasks submitted.")
print("Monitor completion in the EE Tasks tab.")

## Extracting AEF

In [ ]:
import pandas as pd
import os
import glob

# ==========================================================
# Paths
# ==========================================================

drive_folder = (
    str(EMBEDDING_BATCH_DIR)
)

# ==========================================================
# Find all batch CSVs
# ==========================================================

csv_files = sorted(
    glob.glob(
        os.path.join(
            drive_folder,
            "GridEmbeddings_Batch_*.csv"
        )
    )
)

print(f"Found {len(csv_files)} batch files")

# ==========================================================
# Load batches
# ==========================================================

all_batches = []

for file in csv_files:

    try:

        df = pd.read_csv(file)

        all_batches.append(df)

        print(
            f"Loaded {os.path.basename(file)} "
            f"({df.shape[0]:,} rows)"
        )

    except Exception as e:

        print(
            f"Error reading "
            f"{os.path.basename(file)}"
        )

        print(e)

# ==========================================================
# Merge
# ==========================================================

grid_embeddings_df = pd.concat(
    all_batches,
    ignore_index=True
)

print(
    "\nMerged DataFrame shape:",
    grid_embeddings_df.shape
)

# ==========================================================
# Cleanup
# ==========================================================

if 'grid_id' in grid_embeddings_df.columns:

    grid_embeddings_df['grid_id'] = (
        grid_embeddings_df['grid_id']
        .astype(str)
    )

# ==========================================================
# Save final file
# ==========================================================

output_file = os.path.join(
    drive_folder,
    "grid_embeddings_full.csv"
)

grid_embeddings_df.to_csv(
    output_file,
    index=False
)

print(
    f"\nSaved merged file:\n{output_file}"
)

display(grid_embeddings_df.head())

### Bootstrap for Prediction Uncertainty

To quantify the uncertainty of the predictions for each grid point, we'll implement a bootstrap resampling approach. This involves:

1.  **Resampling** the cleaned measured field data (`df_final_training` for each target) with replacement multiple times (e.g., 100 times).
2.  **Training** a separate Random Forest Regressor on each bootstrapped sample.
3.  **Predicting** the target variable for *all* grid points (`final_grid_predictions_df`) using each of the bootstrapped models.
4.  **Calculating the standard deviation** (or other uncertainty metrics like confidence intervals) across all bootstrap predictions for each grid point. This standard deviation will serve as an estimate of the prediction uncertainty.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# ==========================================================
# Definitions from previous cells to ensure scope
# ==========================================================
TARGETS_WANTED = [
    "Volume",   # standing volume
    "n_trees",  # tree density
    "max_h"     # canopy height
]

OUTLIER_SD_THRESH = 0.8   # residual SD threshold for outlier removal

def _sort_A(cols):
    return sorted(
        cols,
        key=lambda x: int(x[1:]) if isinstance(x, str) and x[1:].isdigit() else 10**9
    )

# AEF_COLS will be defined using df_measured_full after it's loaded

# ==========================================================
# Helper function to get cleaned training data (adapted from FQXoCsy8oC4A)
# ==========================================================
def run_cv_for_outlier_detection_for_bootstrap(
    df_full, target, features, outlier_sd_thresh
):
    """Use the same four-region spatial CV to obtain OOF residuals."""
    fold_arr, residuals, df_filtered, _, _ = run_cv(
        df_full, target, features
    )
    threshold = residuals.mean() + outlier_sd_thresh * residuals.std()
    df_clean = df_filtered.loc[residuals <= threshold].copy()
    return df_clean


# ==========================================================
# Bootstrap Function
# ==========================================================
def run_rf_bootstrap_uncertainty(
    df_measured_full, target, aef_cols, outlier_sd_thresh, X_grid_features, n_bootstraps=100, random_state_seed=42
):
    """
    Performs bootstrap resampling to estimate prediction uncertainty.

    Args:
        df_measured_full (pd.DataFrame): DataFrame containing all measured data (including target).
        target (str): The name of the target variable to predict.
        aef_cols (list): List of AEF embedding column names to use as features.
        outlier_sd_thresh (float): Standard deviation threshold for outlier removal.
        X_grid_features (pd.DataFrame): Features for the grid points where predictions are desired.
        n_bootstraps (int): Number of bootstrap iterations.
        random_state_seed (int): Seed for reproducibility.

    Returns:
        (np.ndarray, np.ndarray): Tuple of (mean_predictions, std_predictions) for each grid point.
    """

    print(f"  Starting bootstrap for target: {target}")

    # Get the cleaned training data once for this target (similar to how the final model is trained)
    df_final_training = run_cv_for_outlier_detection_for_bootstrap(df_measured_full, target, aef_cols, outlier_sd_thresh)

    if df_final_training.empty:
        print(f"    Skipping bootstrap for {target}: Cleaned training data is empty.")
        return None, None

    # Store all bootstrap predictions for this target
    all_bootstrap_predictions = np.zeros((X_grid_features.shape[0], n_bootstraps))

    # --- Main Bootstrap Loop ---
    for b in range(n_bootstraps):
        # Resample df_final_training with replacement
        # Use a distinct random state for each bootstrap sample for variety
        bootstrap_sample = df_final_training.sample(
            n=len(df_final_training), replace=True, random_state=random_state_seed + b
        )

        X_train_b = bootstrap_sample[aef_cols]
        y_train_b = bootstrap_sample[target]

        # Fit imputer on the current bootstrap training data
        imputer_b = SimpleImputer(strategy="median")
        X_train_imputed_b = imputer_b.fit_transform(X_train_b)

        # Apply the *same* imputer to the grid features
        X_grid_imputed_b = imputer_b.transform(X_grid_features)

        # Train Random Forest model on the current bootstrap sample
        model_b = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
        model_b.fit(X_train_imputed_b, y_train_b)

        # Predict on the entire grid
        all_bootstrap_predictions[:, b] = model_b.predict(X_grid_imputed_b)

    # Calculate mean and standard deviation across bootstrap predictions
    mean_predictions = np.mean(all_bootstrap_predictions, axis=1)
    std_predictions = np.std(all_bootstrap_predictions, axis=1)
    return mean_predictions, std_predictions

# ==========================================================
# Main execution for bootstrap
# ==========================================================

print("Starting bootstrap uncertainty calculation...")

# Define the full measured dataset from previous context
df_measured_full = merged_embeddings_field_dem_bioclim[
    merged_embeddings_field_dem_bioclim['plot_status'] == 'measured'
].copy()

# Define AEF_COLS using df_measured_full
AEF_COLS = _sort_A([
    c for c in df_measured_full.columns
    if len(c) == 3 and c[0] == "A" and c[1:].isdigit()
])

# Features for the grid points (from previous cell Fl_Kj0k2dKWd)
X_grid_features = grid_embeddings_df[AEF_COLS]

# Initialize final_grid_predictions_df with the grid data BEFORE adding predictions
# This ensures it has the correct number of rows (600614) to match std_dev_uncertainty
final_grid_predictions_df = grid_embeddings_df.copy()

# Number of bootstrap iterations
N_BOOTSTRAPS = 30 # Can be increased for more robust uncertainty estimates

for target in TARGETS_WANTED:
    mean_predictions, std_dev_uncertainty = run_rf_bootstrap_uncertainty(
        df_measured_full=df_measured_full,
        target=target,
        aef_cols=AEF_COLS,
        outlier_sd_thresh=OUTLIER_SD_THRESH,
        X_grid_features=X_grid_features,
        n_bootstraps=N_BOOTSTRAPS
    )

    if mean_predictions is not None and std_dev_uncertainty is not None:
        # Add prediction to the final_grid_predictions_df
        final_grid_predictions_df[f'predicted_{target.lower()}'] = mean_predictions
        print(f"  Added predictions for {target} to final_grid_predictions_df.")

        # Add uncertainty to the final_grid_predictions_df
        final_grid_predictions_df[f'uncertainty_{target.lower()}'] = std_dev_uncertainty
        print(f"  Added uncertainty for {target} to final_grid_predictions_df.")

print("\nBootstrap uncertainty calculation complete.")

# Display head of the updated DataFrame with new uncertainty columns
display(final_grid_predictions_df.head())

# Display descriptive statistics for the uncertainty values
print("\nDescriptive statistics for prediction and uncertainty values:")
output_cols = [
    col for t in TARGETS_WANTED
    for col in [f'predicted_{t.lower()}', f'uncertainty_{t.lower()}']
    if col in final_grid_predictions_df.columns
]
if output_cols:
    display(final_grid_predictions_df[output_cols].describe())
else:
    print("No prediction or uncertainty columns found.")

# Save the updated grid predictions with uncertainty
output_grid_csv_path_with_uncertainty = os.path.join(drive_folder, "grid_predictions_aef_with_uncertainty.csv")
final_grid_predictions_df.to_csv(output_grid_csv_path_with_uncertainty, index=False)
print(f"\nUpdated grid predictions with uncertainty saved to: {output_grid_csv_path_with_uncertainty}")

### Save Predictions with Uncertainty as GeoPackage

In [ ]:
import geopandas as gpd
import os

# Define the folder in Google Drive where files are saved
drive_folder = str(EMBEDDING_BATCH_DIR)

# List of target variables (assuming TARGETS_WANTED is available from previous cells)
targets_for_rounding = ['Volume', 'n_trees', 'max_h']

# Identify prediction and uncertainty columns to round
columns_to_round = []
for target in targets_for_rounding:
    columns_to_round.append(f'predicted_{target.lower()}')
    columns_to_round.append(f'uncertainty_{target.lower()}')

# Create a copy to avoid modifying the original DataFrame if it's used elsewhere
# and round the specified columns to two decimal places
final_grid_predictions_df_rounded = final_grid_predictions_df.copy()
for col in columns_to_round:
    if col in final_grid_predictions_df_rounded.columns:
        final_grid_predictions_df_rounded[col] = final_grid_predictions_df_rounded[col].round(2)

# Create a GeoDataFrame from final_grid_predictions_df (which now includes uncertainty)
# Use original 'longitude' and 'latitude' columns for geometry
grid_gdf_with_uncertainty = gpd.GeoDataFrame(
    final_grid_predictions_df_rounded, # Use the rounded DataFrame here
    geometry=gpd.points_from_xy(final_grid_predictions_df_rounded['longitude'], final_grid_predictions_df_rounded['latitude']),
    crs="EPSG:4326"
)

# Define the output path for the GeoPackage
output_grid_gpkg_path_with_uncertainty = os.path.join(drive_folder, "grid_predictions_aef_with_uncertainty.gpkg")

# Save as GeoPackage
grid_gdf_with_uncertainty.to_file(output_grid_gpkg_path_with_uncertainty, driver="GPKG")

print(f"Grid predictions with uncertainty saved as GeoPackage successfully to: {output_grid_gpkg_path_with_uncertainty}")